# A9-CWO · two-channel Discord daily collector

**Current date-enumerated revision: `20261009-013` (9 October 2026, Nepal time).** The Colab/GitHub working URL remains unchanged. The first complete provider attempt processed 26 sources and 174 changed messages but ended PARTIAL with 749 attachment byte-length mismatches. The preceding `20261009-004` revision added identity-encoded CDN byte diagnostics. Revision `20261009-005` added **stable named cells C01 / C02** and Setup stage markers. Revision **`20261009-006`** adds an optional **C03 read-only single-attachment probe** for the repeating Discord JPG byte-size mismatch. It compares the source and proxied CDN responses without uploading either, advancing any cursor, or claiming original fidelity. Mismatched files are never marked original or complete; the prior version 20261009-003 was snapshotted first. Before **every** later change—even a typo, comment, or other minor edit—archive the current working notebook and assign the next `YYYYMMDD-NNN` revision. Earlier versions: [notebook version manifest](notebook-versions.json) and [numbered archive](99_VERSION_ARCHIVE/README.md).


**Permanent executable cell directory (include the ID when reporting a problem):**

| ID | Name | What it does |
| --- | --- | --- |
| **C01** | **SETUP** | Installs dependencies, authorizes Google Drive, reads the Discord bot token from Colab Secrets |
| **C02** | **PREFLIGHT + CONFIRMED COLLECTOR** | Shows live source roster, Discord links, NPT/UTC windows, cursor/catch-up reason; archives only after operator types `COLLECT` |
| **C03** | **CDN PROBE** | Independently diagnoses one failed JPEG from the last frozen receipt, without modifying Drive or collector state |
| **C04** | **DERIVATIVE PILOT** | Optionally save exactly one image as a labelled private provider derivative after C03; not original certification |
| **C05** | **DAILY INDEX** | Verifies frozen C02 receipt and writes private day-pointer JSONL and immutable run manifest |

If Colab says **Connecting** or **Resuming execution** and there is no `[A9-CWO C01 ...] START` output yet, **the Python cell has not started**; this is a Colab runtime connection issue, not proof of a bad bot token. Reconnect the Colab runtime and retry **C01** only. Never paste the bot token into a chat or GitHub. On an authorized setup, report the last printed **C01 stage** or **C02 stage** and the error without exposing credentials.

**Scope:** exactly guild \`682670494658330644\`, channels \`1463826087841890417\` and \`1492168056858738809\`, and currently active threads whose \`parent_id\` equals one of these two channels. The bot reads no other guild messages.

Run this notebook **before** the A9-CWO daily reconciliation. Its outputs are **raw UTF-8 JSONL, unchanged image/file bytes, machine-readable candidate observations, state cursors, and a JSON handoff receipt** in the established A9 Discord Drive domain. It does **not** post to Discord, Slack or GitHub, and does not call work DONE merely because someone sent a message.

**Setup once:** add the existing authorized Discord bot to both channels with View Channel and Read Message History, enable Message Content in the Discord Developer Portal if required, create a Google Colab secret named \`DISCORD_BOT_TOKEN\`, and grant that Colab notebook access to the secret. Don't use a Discord user token/self-bot. Use the same Google account that owns the A9 Drive tree.

**First run:** a bounded 7-day bootstrap, not full-server history. Each run always revisits seven rolling 24-hour days, extending backwards if a verified cursor is older (plus one-day overlap). The scanned window and coverage debts appear in each receipt. Only active child threads are included in v1: archived threads and image embeds without actual attachment objects are **not** claimed complete.

**Existing Drive authority:** \`12_DISCORD_SYSTEM_AND_ARCHIVES\`, stable ID \`1nTFR9xQXNwgkWRrAq1jJw7ph6rKg3C9X\`. The collector creates/reuses just one child folder \`A9_CWO_DAILY_TWO_CHANNELS\`, rather than duplicating the whole-server archive. Preserve the existing Discord full-fidelity archive and its immutable history.

**Run order:** C01 Setup → C02 preflight (read roster and NPT window) → type `COLLECT` to execute, or anything else for preview only → inspect receipt → C05 index → reconcile. If any source returns PARTIAL, don't claim complete capture or advance its watermark.

**Google Drive API compatibility (v1.0.1):** The Python discovery client's list method uses **`pageSize`** and **`pageToken`** (camelCase), not `page_size` or `page_token`. This fixes the TypeError in `children()`. The setup also uses an explicit `httplib2.Http(timeout=120)` transport for Drive requests. If you already opened this notebook in Colab, use the updated GitHub source or paste the corrected calls; an old Colab copy will not update itself.
Revision `20261009-007` adds C04 SINGLE DERIVATIVE PILOT. After running C01 and C03, C04 stores exactly one mismatched but valid full-resolution provider image under the existing PRIVATE Drive run folder as `PROVIDER_DERIVATIVE_NOT_ORIGINAL`, with immutable sidecar hash/readback; it never advances cursors, rewrites originals, or clears failure counts. Do not run C02 just to invoke C04.

Revision 20261009-008: C01 → C02 rescans seven days with downtime catch-up, then C05 creates immutable private daily pointer indexes. C03/C04 optional; they do not certify original attachments.

Revision `20261009-009` adds a no-archive-write C02 preflight table of every live source and exact NPT/UTC window, with an operator confirmation gate, receipt-preserved plan, and per-source fetch timestamps. It does NOT extend scope to archived Discord threads or resolve prior media integrity debt.

Revision `20261009-010` adds a C02→C05 same-runtime safety gate: preview-only invalidates any earlier confirmed run marker; C05 accepts only the exact freshly confirmed run ID and readback-verified receipt ID. It cannot silently reuse an older provider handoff.

Revision `20261009-011`: optional C06 full-guild historical capture with explicit preflight, full accessible channel/active+archived-thread discovery, unlimited historical age, bounded resumable immutable gzip JSONL Drive pages and per-page original-attachment evidence. The existing daily C02/C05 remain unchanged and DO NOT certify completeness of C06.

Revision `20261009-012` adds C07 original-media repair, C08 SHA-verified normalized SQLite private index, and C09 fail-closed Drive-first website/Slack publication readiness checks. Source facts still require explicit classification and publishing authorization.

Revision `20261009-013`: media reuse is certified only when a fresh authorized Discord CDN download has the correct declared original byte length and its SHA-256 exactly matches the complete private Drive byte readback. A mere file-size match can no longer produce ORIGINAL_REUSED.


## C01 — SETUP · Google Drive + Discord credential

Run **C01** first; wait for **`[A9-CWO C01 SETUP | 06/06] PASS`** before running C02. The outputs are labelled **01/06–06/06**. If the notebook status still shows **Connecting / Resuming execution** and you cannot see **01/06 START**, Colab has not started Python yet—reconnect the runtime instead of changing the bot token. If stuck at **03/06 GOOGLE_AUTH_WAIT**, complete the Google permission dialog. If stuck after **05/06**, grant notebook access to the `DISCORD_BOT_TOKEN` Colab Secret. Never share the actual token.


In [ ]:
# A9-CWO | C01 — SETUP (stable cell ID; revision 20261009-008)
print("[A9-CWO C01 SETUP | 01/06] START — Python cell running; checking dependencies", flush=True)
%pip -q install google-api-python-client google-auth-httplib2 google-auth-oauthlib requests httplib2
print("[A9-CWO C01 SETUP | 02/06] DEPENDENCIES_READY — waiting for Google authorization", flush=True)

from google.colab import auth, userdata
print("[A9-CWO C01 SETUP | 03/06] GOOGLE_AUTH_WAIT — approve the Google dialog if prompted", flush=True)
auth.authenticate_user()
print("[A9-CWO C01 SETUP | 04/06] GOOGLE_AUTH_OK — creating Google Drive client", flush=True)
import google.auth
from googleapiclient.discovery import build
from google_auth_httplib2 import AuthorizedHttp
import httplib2
from getpass import getpass

SCOPES = ["https://www.googleapis.com/auth/drive"]
credentials, _ = google.auth.default(scopes=SCOPES)
# httplib2 does not implement per-request timeout; set it on the transport once.
# Providing AuthorizedHttp avoids passing both 'credentials' and 'http' to build().
authorized_http = AuthorizedHttp(credentials, http=httplib2.Http(timeout=120))
drive = build("drive", "v3", http=authorized_http, cache_discovery=False)
print("[A9-CWO C01 SETUP | 05/06] DRIVE_CLIENT_READY — reading Discord token from Colab Secrets", flush=True)

GUILD_ID = "682670494658330644"
ROOT_DRIVE_ID = "1nTFR9xQXNwgkWRrAq1jJw7ph6rKg3C9X"
PARENT_CHANNEL_IDS = ["1463826087841890417", "1492168056858738809"]
RUN_FOLDER_NAME = "A9_CWO_DAILY_TWO_CHANNELS"
BOOTSTRAP_DAYS = 7
REVISIT_DAYS = 7
BACKFILL_OVERLAP_DAYS = 1
MAX_PAGES_PER_SOURCE = 100
DOWNLOAD_ATTACHMENTS = True
MAX_ATTACHMENT_BYTES = 100 * 1024 * 1024
INCLUDE_ACTIVE_CHILD_THREADS = True

try:
    BOT_TOKEN = userdata.get("DISCORD_BOT_TOKEN")
except Exception:
    BOT_TOKEN = None
if not BOT_TOKEN:
    BOT_TOKEN = getpass("Paste Discord BOT token (input hidden): ").strip()
if not BOT_TOKEN:
    raise RuntimeError("Discord bot token is required. Stop; do not substitute a user token.")
assert len(set(PARENT_CHANNEL_IDS)) == 2
print("[A9-CWO C01 SETUP | 06/06] PASS — Drive client ready; Discord scope: 1 guild, 2 parents. Bot token hidden.", flush=True)

## C02 — COLLECTOR · two Discord channels + Drive archival

Only run after **C01 PASS**. Status lines are prefixed **`A9-CWO C02 COLLECTOR`**; report the last line and the **`FINAL RECEIPT`** status. Media-size mismatches remain **PARTIAL**, not successfully archived originals. This run only reads the two configured parent channels and currently active child threads; it does not post to Discord or Slack.

From revision 20261009-008, run C02 each day for at least the previous seven days, and C05 after new C02 receipt for private machine-readable daily index partitions. Older two-day receipts cannot pass C05.

**r009 TRANSPARENT GATE:** Running C02 first performs live read-only metadata/state discovery and prints every source, parent, ID, Discord link, exact NPT window and catch-up reason. To proceed with archive writes type `COLLECT`. Other input ends as PREVIEW ONLY with no new source receipt. Run C05 only after the confirmed collection.


In [ ]:
# A9-CWO | C02 — COLLECTOR (stable cell ID; revision 20261009-010)
# Read-only Discord API; write raw/source receipts to the established A9 Drive tree
import io, os, re, time, json, hashlib, tempfile, threading
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from urllib.parse import urlparse
import requests
from googleapiclient.http import MediaIoBaseUpload, MediaFileUpload, MediaIoBaseDownload

TZ = ZoneInfo("Asia/Kathmandu")
API = "https://discord.com/api/v10"
http = requests.Session()
http.headers.update({"Authorization": "Bot " + BOT_TOKEN, "User-Agent": "A9-CWO-DailyTwoChannelCollector/1.0"})
RUN_AT = datetime.now(timezone.utc)
RUN_ID = RUN_AT.strftime("%Y%m%dT%H%M%SZ")
# A preview-only C02 execution MUST invalidate any earlier same-kernel C02 approval.
C02_CONFIRMED_RUN_ID = None
C02_CONFIRMED_RECEIPT_ID = None
DISCORD_EPOCH_MS = 1420070400000

# Visible progress and a 20-second heartbeat even while a provider request is blocked.
# The progress stage is diagnostic only; it never changes source cursors or truth.
_RUN_STEP = {"name": "Initializing collector"}
_RUN_STARTED = time.monotonic()
_HEARTBEAT_STOP = threading.Event()

def progress(message):
    _RUN_STEP["name"] = str(message)
    elapsed = int(time.monotonic() - _RUN_STARTED)
    now = datetime.now(TZ).strftime("%H:%M:%S")
    print("[A9-CWO C02 COLLECTOR | %s NPT | +%ds] %s" % (now, elapsed, message), flush=True)

def _heartbeat_loop():
    while not _HEARTBEAT_STOP.wait(20):
        elapsed = int(time.monotonic() - _RUN_STARTED)
        print("[A9-CWO C02 COLLECTOR | heartbeat +%ds] Still running: %s" % (elapsed, _RUN_STEP["name"]), flush=True)

_HEARTBEAT_THREAD = threading.Thread(target=_heartbeat_loop, daemon=True)
_HEARTBEAT_THREAD.start()
progress("Starting. Scope: Discord guild %s, two configured channels; no whole-server harvest" % GUILD_ID)

def jsonbytes(value):
    return (json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")) + "\n").encode("utf-8")

def get_discord(path, params=None):
    for retry in range(6):
        res = http.get(API + path, params=params, timeout=75)
        if res.status_code == 429:
            wait = float(res.json().get("retry_after", 1))
            progress("Discord rate limit on %s; waiting %.1f seconds" % (path, wait))
            time.sleep(min(max(wait, 0.2), 30))
            continue
        if 500 <= res.status_code < 600 and retry < 5:
            progress("Discord HTTP %d from %s; retry %d" % (res.status_code, path, retry + 1))
            time.sleep(min(2**retry, 16))
            continue
        if not res.ok:
            raise RuntimeError("Discord API returned HTTP %s for %s; %s" % (
                res.status_code, path, res.text[:250]))
        return res.json()
    raise RuntimeError("Discord API retries exhausted for " + path)

def after_snowflake(dt):
    ms = int(dt.timestamp() * 1000)
    return str(max(0, ms - DISCORD_EPOCH_MS) << 22)

def quote_drive(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")

def children(parent, name):
    query = "trashed = false and '%s' in parents and name = '%s'" % (quote_drive(parent), quote_drive(name))
    rows = []
    token = None
    while True:
        result = drive.files().list(
            q=query, fields="nextPageToken,files(id,name,mimeType,size,md5Checksum)",
            pageSize=100, pageToken=token, supportsAllDrives=True,
            includeItemsFromAllDrives=True).execute()
        rows += result.get("files", [])
        token = result.get("nextPageToken")
        if not token:
            break
    if len(rows) > 1:
        raise RuntimeError("Ambiguous duplicate Drive name in parent %s: %s" % (parent, name))
    return rows[0] if rows else None

def directory(parent, name):
    progress("Drive: resolve folder %s" % name)
    old = children(parent, name)
    if old:
        if old["mimeType"] != "application/vnd.google-apps.folder":
            raise RuntimeError("Expected folder, found existing file: " + name)
        return old["id"]
    progress("Drive: creating folder %s" % name)
    return drive.files().create(
        body={"name": name, "mimeType": "application/vnd.google-apps.folder",
              "parents": [parent]},
        fields="id", supportsAllDrives=True).execute()["id"]

def byte_readback(file_id):
    stream = io.BytesIO()
    request = drive.files().get_media(fileId=file_id, supportsAllDrives=True)
    dl = MediaIoBaseDownload(stream, request)
    done = False
    while not done:
        _, done = dl.next_chunk()
    return stream.getvalue()

def save_bytes(parent, name, contents, mime="application/json"):
    progress("Drive: checking archival object %s (%d bytes)" % (name, len(contents)))
    expected = hashlib.sha256(contents).hexdigest()
    old = children(parent, name)
    if old:
        actual = byte_readback(old["id"])
        if hashlib.sha256(actual).hexdigest() != expected:
            raise RuntimeError("Existing Drive artifact conflicts; no overwrite: " + name)
        return {"id": old["id"], "sha256": expected, "bytes": len(contents), "reused": True}
    media = MediaIoBaseUpload(io.BytesIO(contents), mimetype=mime, resumable=True)
    progress("Drive: uploading %s" % name)
    obj = drive.files().create(
        body={"name": name, "parents": [parent]}, media_body=media,
        fields="id,size,md5Checksum", supportsAllDrives=True).execute()
    actual = byte_readback(obj["id"])
    if hashlib.sha256(actual).hexdigest() != expected:
        raise RuntimeError("Drive byte readback mismatch for " + name)
    progress("Drive: archived and SHA-256 readback verified: %s" % name)
    return {"id": obj["id"], "sha256": expected, "bytes": len(contents), "reused": False}

def state_read(root):
    progress("Drive: loading existing per-source cursors")
    obj = children(root, "state_current.json")
    if not obj:
        return {"schema": "a9-cwo-discord-state-v1", "guild_id": GUILD_ID,
                "source_states": {}}, None
    parsed = json.loads(byte_readback(obj["id"]).decode("utf-8"))
    if parsed.get("guild_id") != GUILD_ID or parsed.get("schema") != "a9-cwo-discord-state-v1":
        raise RuntimeError("Existing state conflicts with requested guild/schema")
    return parsed, obj

def state_write(root, state, old):
    progress("Drive: saving verified per-channel cursor")
    payload = jsonbytes(state)
    media = MediaIoBaseUpload(io.BytesIO(payload), mimetype="application/json", resumable=True)
    if old:
        obj = drive.files().update(fileId=old["id"], media_body=media,
                                   fields="id", supportsAllDrives=True).execute()
    else:
        obj = drive.files().create(body={"name": "state_current.json", "parents": [root]},
                                   media_body=media, fields="id", supportsAllDrives=True).execute()
    if hashlib.sha256(byte_readback(obj["id"])).digest() != hashlib.sha256(payload).digest():
        raise RuntimeError("State readback failed")
    return {"id": obj["id"]}

def media_archive(folder, msg, attachment):
    aid, mid = str(attachment["id"]), str(msg["id"])
    size = int(attachment.get("size") or 0)
    if size > MAX_ATTACHMENT_BYTES:
        raise RuntimeError("Attachment exceeds MAX_ATTACHMENT_BYTES: id " + aid)
    safe_name = re.sub(r"[^-._a-zA-Z0-9]+", "_", attachment.get("filename", "file"))[:96]
    name = mid + "__" + aid + "__" + safe_name
    old = children(folder, name)
    if old:
        if size and int(old.get("size") or -1) != size:
            raise RuntimeError("Existing attachment size conflict: id " + aid)
        return {"attachment_id": aid, "message_id": mid, "drive_id": old["id"],
                "provider_size": size, "reused": True, "status": "PREVIOUSLY_ARCHIVED"}
    url = attachment.get("url")
    if not url or urlparse(url).scheme != "https":
        raise RuntimeError("Attachment URL missing or non-HTTPS: " + aid)
    progress("Media: message %s attachment %s, %d bytes; checking CDN" % (mid, aid, size))
    # CDN signatures may expire. The next run re-fetches message metadata.
    # A NEW session-less request prevents leaking the Discord Bot Authorization header to a CDN.
    host = (urlparse(url).hostname or "").lower()
    if not (host == "cdn.discordapp.com" or host == "media.discordapp.net" or host.endswith(".discordapp.net")):
        raise RuntimeError("Unexpected attachment CDN hostname for id " + aid)
    # Request unencoded source bytes when the CDN honors identity encoding.
    with requests.get(url, stream=True, timeout=120,
                      headers={"User-Agent": "A9-CWO/1.0",
                               "Accept-Encoding": "identity",
                               "Accept": "application/octet-stream, image/*;q=0.9, */*;q=0.5"}) as res:
        if not res.ok:
            raise RuntimeError("Attachment GET failed: HTTP %s, id %s" % (res.status_code, aid))
        with tempfile.NamedTemporaryFile(suffix=".bin") as tmp:
            response_type = (res.headers.get("Content-Type") or "unknown").split(";")[0].lower().strip()
            response_encoding = res.headers.get("Content-Encoding", "identity")
            reported_length = res.headers.get("Content-Length", "unknown")
            effective_host = urlparse(res.url).hostname or "unknown"
            first_bytes = b""
            sha = hashlib.sha256()
            md5 = hashlib.md5()
            count = 0
            for chunk in res.iter_content(chunk_size=1024*1024):
                if not chunk:
                    continue
                if len(first_bytes) < 16:
                    first_bytes = (first_bytes + chunk)[:16]
                count += len(chunk)
                if count > MAX_ATTACHMENT_BYTES:
                    raise RuntimeError("Attachment size cap exceeded: " + aid)
                sha.update(chunk)
                md5.update(chunk)
                tmp.write(chunk)
            tmp.flush()
            if size and count != size:
                # Log safe evidence only: no signed query parameters or bot credentials.
                # Mismatched bytes must NOT be uploaded as a verified original.
                raise RuntimeError(
                    "Attachment size mismatch, id=%s declared=%d observed=%d "
                    "content_type=%s content_length_header=%s encoding=%s host=%s "
                    "first16_hex=%s" % (
                        aid, size, count, response_type,
                        reported_length, response_encoding, effective_host,
                        first_bytes.hex()))
            expected_type = (attachment.get("content_type") or "").split(";")[0].lower().strip()
            if expected_type.startswith("image/") and not response_type.startswith("image/"):
                raise RuntimeError(
                    "Attachment MIME mismatch id=%s declared_mime=%s received_mime=%s first16_hex=%s" %
                    (aid, expected_type, response_type, first_bytes.hex()))
            media = MediaFileUpload(tmp.name, mimetype=attachment.get("content_type") or "application/octet-stream",
                                    resumable=True)
            progress("Media: upload attachment %s to Drive" % aid)
            created = drive.files().create(body={"name": name, "parents": [folder]},
                                           media_body=media, fields="id,size,md5Checksum",
                                           supportsAllDrives=True).execute()
    if int(created.get("size") or -1) != count or (created.get("md5Checksum") and created["md5Checksum"] != md5.hexdigest()):
        raise RuntimeError("Provider size/MD5 readback mismatch: attachment " + aid)
    progress("Media: stored and verified attachment %s (%d bytes)" % (aid, count))
    return {"attachment_id": aid, "message_id": mid, "drive_id": created["id"],
            "provider_size": size, "stored_bytes": count, "sha256": sha.hexdigest(),
            "provider_md5": created.get("md5Checksum"), "reused": False, "status": "ARCHIVED"}

def fetch_pages(channel_id, lower_bound):
    collected, before = [], None
    for page_number in range(MAX_PAGES_PER_SOURCE):
        if page_number == 0 or page_number % 5 == 0:
            progress("Discord: channel %s, fetching page %d, collected %d messages" %
                     (channel_id, page_number + 1, len(collected)))
        params = {"limit": 100}
        if before:
            params["before"] = before
        page = get_discord("/channels/%s/messages" % channel_id, params)
        if not page:
            return collected
        collected.extend(m for m in page if int(m["id"]) > int(lower_bound))
        earliest = min(int(m["id"]) for m in page)
        if earliest <= int(lower_bound) or len(page) < 100:
            return collected
        before = str(earliest)
    raise RuntimeError("MAX_PAGES_PER_SOURCE exceeded; cursor will not advance: " + channel_id)

def in_scope_sources():
    progress("Discord: checking permissions/metadata for exactly two parent channels")
    scoped = {}
    for parent_id in PARENT_CHANNEL_IDS:
        meta = get_discord("/channels/" + parent_id)
        if str(meta.get("guild_id")) != GUILD_ID:
            raise RuntimeError("Channel not in configured guild: " + parent_id)
        scoped[parent_id] = {"parent_id": parent_id, "channel_id": parent_id, "name": meta.get("name"), "type": meta.get("type")}
    if INCLUDE_ACTIVE_CHILD_THREADS:
        progress("Discord: discovering active child-thread metadata; parent channels remain restricted")
        # This endpoint reads THREAD METADATA, not messages in unrelated channels.
        active = get_discord("/guilds/%s/threads/active" % GUILD_ID)
        for thread in active.get("threads", []):
            parent_id = str(thread.get("parent_id"))
            thread_id = str(thread.get("id"))
            if parent_id in PARENT_CHANNEL_IDS:
                scoped[thread_id] = {
                    "parent_id": parent_id, "channel_id": thread_id,
                    "name": thread.get("name"), "type": thread.get("type")}
    progress("Discord: %d scoped parent/active-thread sources found" % len(scoped))
    return scoped


# PURE READ-ONLY PLANNER — no provider calls or Drive mutations.
def build_run_plan(scoped_sources, archived_state, planned_at):
    baseline = planned_at - timedelta(days=REVISIT_DAYS)
    rows = []
    for source_id, meta in sorted(scoped_sources.items()):
        saved = archived_state.get("source_states", {}).get(source_id, {})
        cursor = saved.get("highwater_message_id")
        cutoff, mode, cursor_time = baseline, "ROLLING_7_DAYS_NO_VERIFIED_CURSOR", None
        if cursor:
            cursor_ms = (int(cursor) >> 22) + DISCORD_EPOCH_MS
            cursor_dt = datetime.fromtimestamp(cursor_ms / 1000, timezone.utc)
            cursor_time = cursor_dt.isoformat()
            overlap_start = cursor_dt - timedelta(days=BACKFILL_OVERLAP_DAYS)
            if overlap_start < cutoff:
                cutoff, mode = overlap_start, "CATCHUP_OLDER_CURSOR_PLUS_1_DAY_OVERLAP"
            else:
                mode = "ROLLING_7_DAYS_VERIFIED_CURSOR"
        parent = str(meta["parent_id"])
        rows.append({
            "channel_id": source_id,
            "channel_name": meta.get("name") or "(unnamed)",
            "parent_channel_id": parent,
            "parent_channel_name": scoped_sources[parent].get("name") or "(unnamed parent)",
            "kind": "PARENT" if source_id == parent else "ACTIVE_THREAD",
            "window_start_utc": cutoff.isoformat(),
            "window_start_npt": cutoff.astimezone(TZ).isoformat(),
            "planned_start_utc": planned_at.isoformat(),
            "planned_start_npt": planned_at.astimezone(TZ).isoformat(),
            "window_rule": mode,
            "verified_cursor": cursor,
            "verified_cursor_at_utc": cursor_time,
            "discord_url": "https://discord.com/channels/%s/%s" % (GUILD_ID, source_id),
        })
    return rows

def show_run_plan(rows):
    print("\n" + "=" * 78, flush=True)
    print("A9-CWO C02: BEFORE COLLECTION — LIVE SOURCE PLAN (READ-ONLY)", flush=True)
    print("Run ID: %s | Run start: %s NPT / %s UTC" %
          (RUN_ID, RUN_AT.astimezone(TZ).isoformat(), RUN_AT.isoformat()), flush=True)
    print("Guild=%s | parents=%d | live scoped sources=%d" %
          (GUILD_ID, len(PARENT_CHANNEL_IDS), len(rows)), flush=True)
    print("Default: past %d x 24 hours. Older verified cursor => %d-day safety overlap." %
          (REVISIT_DAYS, BACKFILL_OVERLAP_DAYS), flush=True)
    print("Source fetch may include messages posted during this run; actual fetch times are recorded.", flush=True)
    print("NOT INCLUDED: Discord-archived child threads, embed-only external media, other guild channels.", flush=True)
    print("CAUTION: failed prior original media may be retried; never treat it as original-complete.", flush=True)
    for idx, item in enumerate(rows, 1):
        print("[%02d/%02d] %s | %s | id=%s | parent=%s" %
              (idx, len(rows), item["kind"], item["channel_name"], item["channel_id"], item["parent_channel_name"]), flush=True)
        print("            %s NPT -> %s NPT | %s" %
              (item["window_start_npt"], item["planned_start_npt"], item["window_rule"]), flush=True)
        print("            cursor=%s | %s" %
              (item["verified_cursor"] or "NONE", item["discord_url"]), flush=True)
    print("No raw messages, attachment URLs or credentials appear in this preview.", flush=True)
    print("=" * 78 + "\n", flush=True)

try:
    progress("PREFLIGHT: reading Drive state and live Discord parent/active-thread metadata")
    root_meta = drive.files().get(fileId=ROOT_DRIVE_ID, fields="id,name,mimeType", supportsAllDrives=True).execute()
    if root_meta["mimeType"] != "application/vnd.google-apps.folder":
        raise RuntimeError("Existing A9 Drive root was not a folder")
    prior_root = children(ROOT_DRIVE_ID, RUN_FOLDER_NAME)
    if prior_root and prior_root["mimeType"] != "application/vnd.google-apps.folder":
        raise RuntimeError("Existing daily intake is not a Drive folder")
    root = prior_root["id"] if prior_root else None
    state, oldstate = state_read(root) if root else (
        {"schema": "a9-cwo-discord-state-v1", "guild_id": GUILD_ID, "source_states": {}}, None)
    scoped = in_scope_sources()
    run_plan = build_run_plan(scoped, state, RUN_AT)
    show_run_plan(run_plan)
    operator_choice = input("Type COLLECT to run the above plan, otherwise preview only: ").strip()
    if operator_choice != "COLLECT":
        progress("PREVIEW ONLY — no archived messages, receipts or cursor state written")
    else:
        if root is None:
            root = directory(ROOT_DRIVE_ID, RUN_FOLDER_NAME)
        fresh_state, fresh_state_file = state_read(root)
        if jsonbytes(fresh_state) != jsonbytes(state):
            raise RuntimeError("Cursor state changed during preview; stop and restart.")
        oldstate = fresh_state_file
        rawdir = directory(root, "RAW_JSONL")
        mediadir = directory(root, "ATTACHMENT_BYTES")
        canddir = directory(root, "CANDIDATES_JSONL")
        receiptdir = directory(root, "RUN_RECEIPTS")
        progress("COLLECT confirmed — proceeding with %d scoped sources" % len(run_plan))
        receipt = {
         "schema": "a9-cwo-discord-receipt-v1", "run_id": RUN_ID, "guild_id": GUILD_ID,
         "target_parent_channels": PARENT_CHANNEL_IDS, "root_drive_id": root,
         "collection_started_at": RUN_AT.isoformat(), "sources": [],
         "scope_rule": "Only selected parent channels and currently active children of those parents",
         "archived_threads_covered": False, "embed_only_images_covered": False,
         "scan_rule": "ROLLING_SEVEN_DAYS_WITH_CATCHUP_EXTENSION",
         "replay_lookback_days": REVISIT_DAYS, "bootstrap_days": BOOTSTRAP_DAYS,
         "catchup_overlap_days": BACKFILL_OVERLAP_DAYS,
         "preflight_confirmed": True, "preflight_plan": run_plan,
         "preflight_generated_at_utc": datetime.now(timezone.utc).isoformat(),
         "preflight_run_start_utc": RUN_AT.isoformat(),
         "preflight_time_note": "Lower bounds frozen at RUN_AT; late messages may be fetched during execution"
        }
        for plan in run_plan:
            source_id = plan["channel_id"]
            meta = scoped[source_id]
            progress("Discord: BEGIN source %s (%s), parent %s" %
                     (source_id, meta.get("name") or "unnamed", meta["parent_id"]))
            record = {"channel_id": source_id, "parent_channel_id": meta["parent_id"],
                      "channel_name": meta["name"], "status": "PENDING"}
            current = dict(state["source_states"].get(source_id, {}))
            last_good = current.get("highwater_message_id")
            # Freeze the lower bound at preflight, not minutes later during the scan.
            horizon = datetime.fromisoformat(plan["window_start_utc"])
            lower = after_snowflake(horizon)
            record["window_start_utc"] = horizon.isoformat()
            record["window_start_npt"] = plan["window_start_npt"]
            record["window_rule"] = plan["window_rule"]
            record["previous_highwater_message_id"] = last_good
            progress("WINDOW source %s: %s NPT -> %s NPT (%s)" %
                     (source_id, plan["window_start_npt"], plan["planned_start_npt"], plan["window_rule"]))
            try:
                record["message_fetch_started_at_utc"] = datetime.now(timezone.utc).isoformat()
                messages = fetch_pages(source_id, lower)
                record["message_fetch_finished_at_utc"] = datetime.now(timezone.utc).isoformat()
                messages.sort(key=lambda m: int(m["id"]))
                progress("Discord: source %s scanned %d messages; processing attachment evidence" %
                         (source_id, len(messages)))
                versions = dict(current.get("recent_versions", {}))
                changed, attachments, candidate_rows = [], [], []
                reused = 0
                folder = directory(mediadir, source_id)
                for msg in messages:
                    key = str(msg["id"])
                    # Do not hash expiring signed CDN URLs: stable content, edits and attachment IDs determine a version.
                    content_signature = {
                        "id": key, "content": msg.get("content"), "edited_timestamp": msg.get("edited_timestamp"),
                        "author_id": msg.get("author", {}).get("id"),
                        "attachments": [{"id": a.get("id"), "size": a.get("size"),
                                         "filename": a.get("filename")} for a in msg.get("attachments", [])],
                        "embeds": msg.get("embeds"), "flags": msg.get("flags"),
                        "reactions": msg.get("reactions")
                    }
                    raw_digest = hashlib.sha256(jsonbytes(content_signature)).hexdigest()
                    if versions.get(key) == raw_digest:
                        reused += 1
                        continue
                    changed.append(msg)
                    for attachment in msg.get("attachments", []):
                        try:
                            attachments.append(media_archive(folder, msg, attachment) if DOWNLOAD_ATTACHMENTS
                                               else {"attachment_id": str(attachment["id"]), "status": "NOT_DOWNLOADED"})
                        except Exception as err:
                            attachments.append({"attachment_id": str(attachment.get("id")),
                                                "message_id": key, "status": "FAILED",
                                                "reason": str(err)[:300]})
                    candidate_rows.append({
                        "provider": "DISCORD", "guild_id": GUILD_ID,
                        "parent_channel_id": meta["parent_id"], "channel_id": source_id,
                        "message_id": key, "message_url": "https://discord.com/channels/%s/%s/%s" % (GUILD_ID, source_id, key),
                        "author_id": msg.get("author", {}).get("id"), "created_at": msg.get("timestamp"),
                        "edited_at": msg.get("edited_timestamp"), "text_original": msg.get("content", ""),
                        "attachment_ids": [str(a.get("id")) for a in msg.get("attachments", [])],
                        "project_id": None, "fact_admission": "NEEDS_RECONCILIATION"
                    })
                    versions[key] = raw_digest
                errors = [a for a in attachments if a.get("status") in ("FAILED", "NOT_DOWNLOADED")]
                record["scanned_messages"] = len(messages)
                record["new_or_changed_messages"] = len(changed)
                record["previously_seen_unchanged"] = reused
                record["attachments"] = len(attachments)
                record["failed_or_unarchived_attachments"] = len(errors)
                record["attachments_evidence"] = attachments
                if errors:
                    for example in errors[:3]:
                        progress("MEDIA FAILURE DETAIL source %s attachment %s: %s" %
                                 (source_id, example.get("attachment_id"), example.get("reason", "NOT_ARCHIVED")))
                    if len(errors) > 3:
                        progress("MEDIA FAILURE DETAIL source %s: %d further failures in frozen receipt" %
                                 (source_id, len(errors)-3))
                progress("Discord: source %s changed %d, unchanged %d, attachments %d, errors %d" %
                         (source_id, len(changed), reused, len(attachments), len(errors)))
                if changed:
                    suffix = source_id + "__" + RUN_ID
                    raw = b"".join(jsonbytes(x) for x in changed)
                    candidates = b"".join(jsonbytes(x) for x in candidate_rows)
                    record["raw_jsonl"] = save_bytes(rawdir, suffix + ".jsonl", raw, "application/x-ndjson")
                    record["candidate_jsonl"] = save_bytes(canddir, suffix + ".jsonl", candidates, "application/x-ndjson")
                if errors:
                    record["status"] = "PARTIAL_ATTACHMENTS"
                else:
                    record["status"] = "PASS"
                    if messages:
                        newest = str(max(int(x["id"]) for x in messages))
                        current["highwater_message_id"] = str(max(int(last_good or 0), int(newest)))
                    current["recent_versions"] = versions
                    current["last_verified_at"] = datetime.now(timezone.utc).isoformat()
                    state["source_states"][source_id] = current
                    # Persist verified checkpoint after each successful source, so later source failure cannot lose progress.
                    oldstate = state_write(root, state, oldstate)
            except Exception as err:
                progress("FAILED source %s: %s" % (source_id, str(err)[:250]))
                record["status"] = "FAILED_SOURCE"
                record["error"] = str(err)[:600]
                # No cursor advance on this source.
            record["source_finished_at_utc"] = datetime.now(timezone.utc).isoformat()
            receipt["sources"].append(record)
            progress("Source %s finished: %s. Captured %d new/changed messages" %
                     (source_id, record["status"], record.get("new_or_changed_messages", 0)))
        receipt["collection_finished_at"] = datetime.now(timezone.utc).isoformat()
        receipt["status"] = "PASS" if receipt["sources"] and all(x["status"] == "PASS" for x in receipt["sources"]) else "PARTIAL"
        receipt["source_count"] = len(receipt["sources"])
        receipt["messages_new_or_changed"] = sum(x.get("new_or_changed_messages", 0) for x in receipt["sources"])
        receipt["attachment_failures"] = sum(x.get("failed_or_unarchived_attachments", 0) for x in receipt["sources"])
        # Frozen run receipt is written only after channel-specific archival attempts.
        progress("Drive: writing frozen run receipt")
        receipt_link = save_bytes(receiptdir, "A9_CWO_DISCORD_RECEIPT__" + RUN_ID + ".json", jsonbytes(receipt))
        handoff = {
         "schema": "a9-cwo-discord-handoff-v1", "provider": "DISCORD",
         "run_id": RUN_ID, "status": receipt["status"],
         "guild_id": GUILD_ID, "scope": PARENT_CHANNEL_IDS,
         "receipt_file_id": receipt_link["id"], "archive_root_drive_id": root,
         "new_or_changed_message_count": receipt["messages_new_or_changed"],
         "collection_finished_at": receipt["collection_finished_at"],
         "per_channel": [{k: item.get(k) for k in (
             "channel_id", "parent_channel_id", "status", "previous_highwater_message_id",
             "scanned_messages", "new_or_changed_messages", "failed_or_unarchived_attachments")}
           for item in receipt["sources"]]
        }
        progress("Drive: publishing and verifying handoff_current.json")
        oldhandoff = children(root, "handoff_current.json")
        media = MediaIoBaseUpload(io.BytesIO(jsonbytes(handoff)), mimetype="application/json", resumable=True)
        if oldhandoff:
            p = drive.files().update(fileId=oldhandoff["id"], media_body=media, fields="id", supportsAllDrives=True).execute()
        else:
            p = drive.files().create(body={"name": "handoff_current.json", "parents": [root]},
                                     media_body=media, fields="id", supportsAllDrives=True).execute()
        if byte_readback(p["id"]) != jsonbytes(handoff):
            raise RuntimeError("Handoff readback failed; A9-CWO is NOT ready")
        # Activate C05 only AFTER this exact run's frozen provider receipt passes readback.
        C02_CONFIRMED_RUN_ID = RUN_ID
        C02_CONFIRMED_RECEIPT_ID = receipt_link["id"]
        print(json.dumps({"status": receipt["status"], "run_id": RUN_ID,
            "scoped_sources": receipt["source_count"], "new_or_changed_messages": receipt["messages_new_or_changed"],
            "attachment_failures": receipt["attachment_failures"], "handoff_drive_id": p["id"],
            "receipt_drive_id": receipt_link["id"], "archive_folder": root}, indent=2))
        progress("FINAL RECEIPT: %s; %d sources; %d new/changed messages; %d attachment failures" %
                 (receipt["status"], receipt["source_count"],
                  receipt["messages_new_or_changed"], receipt["attachment_failures"]))
        print("Run A9-CWO only after checking this receipt. 'PARTIAL' needs reconciliation.", flush=True)
finally:
    _HEARTBEAT_STOP.set()
    progress("Collector execution ended; see FINAL RECEIPT above if it completed successfully")


## C03 — CDN PROBE · one failed Discord attachment (read-only)

**Run C01 first; C02 does not need to be repeated.** This optional C03 cell reads the existing `handoff_current.json` and frozen receipt in the A9 Discord Drive folder, automatically selects the first `Attachment size mismatch` record, refreshes its current Discord message metadata, then downloads at most **two** variants: the attachment's `url` and `proxy_url`. It prints **declared bytes, observed bytes, JPEG dimensions, MIME, Content-Length, SHA-256 and status**. It never prints actual signed URLs, token, or query values; the URL scheme and host are checked. It creates/changes **no Drive files, cursors or original-file claims**. Sharing the output log from C03 (without credentials) is sufficient for the next fix.

**Current unresolved blocker:** run `20261009T020256Z` = **PARTIAL**; 26 sources, 174 changed messages and 749 mismatched attachment bytes. `Accept-Encoding: identity` did not resolve these failures. A valid JPEG response and a matching HTTP Content-Length are insufficient to prove byte fidelity to Discord's attachment `size`. Do not mark these as verified originals.


In [ ]:
# A9-CWO | C03 — CDN PROBE (stable cell ID; revision 20261009-006)
# READ ONLY: one frozen failure; at most one source URL GET and one proxy URL GET.
# Never print Discord auth, signed CDN URLs, signed query values or picture contents.
import io, json, hashlib, tempfile
from urllib.parse import urlparse, parse_qs
import requests
from PIL import Image
from googleapiclient.http import MediaIoBaseDownload

print("[A9-CWO C03 CDN PROBE | 01/05] START — resolving frozen handoff in Drive", flush=True)

def c03_find_child(parent_id, name):
    escaped = str(name).replace("\\", "\\\\").replace("'", "\\'")
    result = drive.files().list(
        q="name = '%s' and '%s' in parents and trashed = false" % (escaped, parent_id),
        fields="nextPageToken,files(id,name,mimeType)", pageSize=100,
        supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
    candidates = result.get("files", [])
    if len(candidates) != 1 or result.get("nextPageToken"):
        raise RuntimeError("C03 needs exactly one named A9 Drive object: " + name)
    return candidates[0]["id"]

def c03_read_json(file_id, max_bytes=12*1024*1024):
    buf = io.BytesIO()
    request = drive.files().get_media(fileId=file_id, supportsAllDrives=True)
    downloader = MediaIoBaseDownload(buf, request, chunksize=1024*1024)
    completed = False
    while not completed:
        _, completed = downloader.next_chunk()
        if buf.tell() > max_bytes:
            raise RuntimeError("C03 JSON safety limit exceeded")
    return json.loads(buf.getvalue().decode("utf-8"))

run_root = c03_find_child(ROOT_DRIVE_ID, RUN_FOLDER_NAME)
handoff = c03_read_json(c03_find_child(run_root, "handoff_current.json"))
if handoff.get("guild_id") != GUILD_ID or handoff.get("status") != "PARTIAL":
    raise RuntimeError("C03 requires a matching frozen PARTIAL Discord handoff")
receipt = c03_read_json(handoff["receipt_file_id"])
if receipt.get("run_id") != handoff.get("run_id") or receipt.get("guild_id") != GUILD_ID:
    raise RuntimeError("C03 receipt/handoff identity mismatch")

chosen = None
for source in receipt.get("sources", []):
    if str(source.get("parent_channel_id")) not in set(PARENT_CHANNEL_IDS):
        continue
    for evidence in source.get("attachments_evidence", []):
        if evidence.get("status") == "FAILED" and "Attachment size mismatch" in evidence.get("reason", ""):
            chosen = (str(source["channel_id"]), evidence)
            break
    if chosen:
        break
if chosen is None:
    raise RuntimeError("C03 did not find a failed attachment in the frozen receipt")
channel_id, evidence = chosen
message_id, attachment_id = str(evidence["message_id"]), str(evidence["attachment_id"])
print("[A9-CWO C03 CDN PROBE | 02/05] FROZEN FAILURE — run=%s channel=%s message=%s attachment=%s" %
      (handoff["run_id"], channel_id, message_id, attachment_id), flush=True)

discord_response = requests.get(
    "https://discord.com/api/v10/channels/%s/messages/%s" % (channel_id, message_id),
    headers={"Authorization": "Bot " + BOT_TOKEN,
             "User-Agent": "A9-CWO-ReadOnlyMediaProbe/1.0"},
    timeout=45)
if not discord_response.ok:
    raise RuntimeError("C03 Discord scoped message refresh HTTP %d" % discord_response.status_code)
message = discord_response.json()
if str(message.get("channel_id")) != channel_id or str(message.get("id")) != message_id:
    raise RuntimeError("C03 refreshed message identity mismatch")
attachment = next((a for a in message.get("attachments", [])
                   if str(a.get("id")) == attachment_id), None)
if attachment is None:
    raise RuntimeError("C03 attachment missing from refreshed Discord message")

expected_size = int(attachment.get("size") or 0)
expected_dims = [attachment.get("width"), attachment.get("height")]
print("[A9-CWO C03 CDN PROBE | 03/05] METADATA — declared_bytes=%d dimensions=%s MIME=%s" %
      (expected_size, expected_dims, attachment.get("content_type", "unknown")), flush=True)

def c03_probe(label, signed_url):
    if not signed_url:
        return {"variant": label, "status": "URL_NOT_PRESENT"}
    parts = urlparse(signed_url)
    allowed = {"cdn.discordapp.com", "media.discordapp.net"}
    if parts.scheme != "https" or parts.hostname not in allowed:
        return {"variant": label, "status": "REJECTED_UNAPPROVED_URL"}
    with requests.get(
        signed_url, stream=True, timeout=(20, 120), allow_redirects=False,
        headers={"User-Agent": "A9-CWO-ReadOnlyMediaProbe/1.0",
                 "Accept-Encoding": "identity",
                 "Accept": "application/octet-stream, image/*;q=0.9, */*;q=0.5"}
    ) as response:
        report = {
            "variant": label, "status": "RESPONSE", "http_code": response.status_code,
            "host": parts.hostname, "query_keys_only": sorted(parse_qs(parts.query)),
            "content_type": response.headers.get("Content-Type", "unknown"),
            "content_encoding": response.headers.get("Content-Encoding", "identity"),
            "content_length": response.headers.get("Content-Length", "unknown"),
            "declared_bytes": expected_size,
            "declared_dimensions": expected_dims
        }
        if not response.ok or 300 <= response.status_code < 400:
            report["status"] = "HTTP_UNUSABLE"
            return report
        count, digest = 0, hashlib.sha256()
        with tempfile.NamedTemporaryFile(suffix=".probe", mode="w+b") as temp:
            for chunk in response.iter_content(chunk_size=1024*1024):
                if not chunk:
                    continue
                count += len(chunk)
                if count > 100*1024*1024:
                    report["status"] = "PROBE_LIMIT_EXCEEDED"
                    return report
                digest.update(chunk)
                temp.write(chunk)
            temp.flush()
            report["observed_bytes"] = count
            report["sha256"] = digest.hexdigest()
            report["byte_count_matches_discord"] = (count == expected_size)
            try:
                with Image.open(temp.name) as img:
                    report["image_format"] = img.format
                    report["observed_dimensions"] = list(img.size)
                    report["dimensions_match_discord"] = list(img.size) == expected_dims
                    img.verify()
                report["image_decodes"] = True
            except Exception as error:
                report["image_decodes"] = False
                report["image_diagnostic"] = type(error).__name__
        report["status"] = "DIAGNOSTIC_ONLY_NOT_CERTIFIED_ORIGINAL"
        return report

print("[A9-CWO C03 CDN PROBE | 04/05] FETCH — exactly one source URL and one proxy URL, read-only", flush=True)
diagnostics = [c03_probe("source_url", attachment.get("url")),
               c03_probe("proxy_url", attachment.get("proxy_url"))]
for item in diagnostics:
    print("[A9-CWO C03 CDN PROBE | RESULT] " + json.dumps(item, sort_keys=True), flush=True)
print("[A9-CWO C03 CDN PROBE | 05/05] DONE — no Drive writes/cursor updates/original certification", flush=True)


## C04 — SINGLE DERIVATIVE PILOT · 1 image, private Drive, original still unresolved

Run **C01 and C03 first** in the same active runtime; then run C04 only if the sample is a valid image with a **confirmed Discord metadata byte-size mismatch**. C04 refuses if the freshly fetched source image differs from C03's SHA-256 or observed bytes. It writes at most one provider representation under the existing A9 intake folder, in a separate `DERIVATIVE_BYTES` directory, with a sidecar under `RUN_RECEIPTS`. It verifies both via Drive readback. Labels are `PROVIDER_DERIVATIVE_NOT_ORIGINAL` and `UNVERIFIED_ORIGINAL_MEDIA_FAILURE_OPEN`. It never changes `handoff_current`, `state_current`, prior run receipts, original attachment folders or failed-source cursors. Source token and signed URLs are never printed. No batch recovery is authorized by this pilot.


In [ ]:
# A9-CWO | C04 — SINGLE DERIVATIVE PILOT (revision 20261009-007)
# Requires C01 and C03 in this runtime. One attachment; ONE source CDN fetch.
# Never marks original complete, edits old archive, or advances any cursor.
import io, json, hashlib
from datetime import datetime, timezone
from urllib.parse import urlparse
import requests
from PIL import Image
from googleapiclient.http import MediaIoBaseUpload, MediaIoBaseDownload

print("[A9-CWO C04 DERIVATIVE PILOT | 01/06] START — validating C03 and frozen receipt", flush=True)
required = ("drive", "BOT_TOKEN", "ROOT_DRIVE_ID", "PARENT_CHANNEL_IDS",
            "run_root", "handoff", "receipt", "chosen", "attachment", "diagnostics")
if any(name not in globals() for name in required):
    raise RuntimeError("C04 requires C01 SETUP then C03 CDN PROBE in this Colab runtime")
if handoff.get("status") != "PARTIAL" or receipt.get("run_id") != handoff.get("run_id"):
    raise RuntimeError("C04 requires matching frozen PARTIAL handoff and receipt")
channel_id, failed = chosen
message_id, attachment_id = str(failed["message_id"]), str(failed["attachment_id"])
source_info = next((d for d in diagnostics if d.get("variant") == "source_url"), None)
if not source_info or source_info.get("status") != "DIAGNOSTIC_ONLY_NOT_CERTIFIED_ORIGINAL":
    raise RuntimeError("C04 source CDN must first be assessed by C03")
if source_info.get("byte_count_matches_discord") or not source_info.get("image_decodes"):
    raise RuntimeError("C04 refuses this sample because it is not a verified mismatch JPEG")
scoped_sources = [s for s in receipt.get("sources", [])
                  if str(s.get("channel_id")) == channel_id
                  and str(s.get("parent_channel_id")) in set(PARENT_CHANNEL_IDS)]
if len(scoped_sources) != 1 or failed.get("status") != "FAILED":
    raise RuntimeError("C04 chosen failure is not uniquely within the authorized two-parent scope")
if str(attachment.get("id")) != attachment_id or not str(attachment.get("content_type", "")).startswith("image/jpeg"):
    raise RuntimeError("C04 refreshed attachment identity/MIME mismatch")
url = attachment.get("url")
parts = urlparse(url or "")
if (parts.scheme != "https" or parts.hostname != "cdn.discordapp.com"
        or not parts.path.startswith("/attachments/%s/%s/" % (channel_id, attachment_id))):
    raise RuntimeError("C04 rejects a source URL outside the chosen Discord attachment")
declared_bytes = int(attachment.get("size") or 0)
declared_dims = [int(attachment.get("width") or 0), int(attachment.get("height") or 0)]
if declared_bytes <= 0 or min(declared_dims) <= 0:
    raise RuntimeError("C04 requires known Discord media metadata")
print("[A9-CWO C04 DERIVATIVE PILOT | 02/06] SOURCE — downloading one bounded image", flush=True)

with requests.get(url, stream=True, allow_redirects=False, timeout=(20, 120),
                  headers={"User-Agent": "A9-CWO-BoundedDerivativePilot/1.0",
                           "Accept-Encoding": "identity"}) as response:
    if response.status_code != 200 or "image/jpeg" not in response.headers.get("Content-Type", ""):
        raise RuntimeError("C04 source image HTTP/MIME failed")
    payload = bytearray()
    for block in response.iter_content(chunk_size=1024 * 1024):
        if block:
            payload.extend(block)
            if len(payload) > 25 * 1024 * 1024:
                raise RuntimeError("C04 25MiB single-image limit reached")
raw = bytes(payload)
digest = hashlib.sha256(raw).hexdigest()
with Image.open(io.BytesIO(raw)) as image:
    actual_dims = list(image.size)
    if image.format != "JPEG" or actual_dims != declared_dims:
        raise RuntimeError("C04 JPEG format or dimensions do not match metadata")
    image.verify()
if len(raw) == declared_bytes:
    raise RuntimeError("C04 refuses to label a same-length image as derivative without further review")
if digest != source_info.get("sha256") or len(raw) != source_info.get("observed_bytes"):
    raise RuntimeError("C04 repeat CDN fetch differs from C03 — representation changed; stop without write")
print("[A9-CWO C04 DERIVATIVE PILOT | 03/06] VALID — JPEG matches dimensions, hash matches C03, original size does NOT match", flush=True)

def c04_match(parent, filename):
    escaped = filename.replace("\\", "\\\\").replace("'", "\\'")
    result = drive.files().list(
        q="name = '%s' and '%s' in parents and trashed = false" % (escaped, parent),
        fields="nextPageToken,files(id,name,size,mimeType)", pageSize=100,
        supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
    matches = result.get("files", [])
    if result.get("nextPageToken") or len(matches) > 1:
        raise RuntimeError("C04 ambiguous Drive object " + filename)
    return matches[0] if matches else None

def c04_read_bytes(fid, max_bytes=26*1024*1024):
    buf = io.BytesIO()
    dl = MediaIoBaseDownload(buf, drive.files().get_media(fileId=fid, supportsAllDrives=True),
                             chunksize=1024*1024)
    done = False
    while not done:
        _, done = dl.next_chunk()
        if buf.tell() > max_bytes:
            raise RuntimeError("C04 Drive readback limit exceeded")
    return buf.getvalue()

folder = c04_match(run_root, "DERIVATIVE_BYTES")
if folder is None:
    folder = drive.files().create(
        body={"name": "DERIVATIVE_BYTES", "mimeType": "application/vnd.google-apps.folder",
              "parents": [run_root]},
        fields="id,name", supportsAllDrives=True).execute()
folder_id = folder["id"]
name = "C04_PROVIDER_DERIVATIVE__%s__%s__source_url__%s.jpg" % (
    handoff["run_id"], attachment_id, digest[:16])
current = c04_match(folder_id, name)
print("[A9-CWO C04 DERIVATIVE PILOT | 04/06] PRIVATE DRIVE — storing one labelled derivative only", flush=True)
if current:
    image_file_id = current["id"]
else:
    image_obj = drive.files().create(
        body={"name": name, "parents": [folder_id],
              "description": "A9-CWO PROVIDER_DERIVATIVE_NOT_ORIGINAL. Original-byte mismatch unresolved."},
        media_body=MediaIoBaseUpload(io.BytesIO(raw), mimetype="image/jpeg", resumable=False),
        fields="id,name,size", supportsAllDrives=True).execute()
    image_file_id = image_obj["id"]
image_readback = c04_read_bytes(image_file_id)
if len(image_readback) != len(raw) or hashlib.sha256(image_readback).hexdigest() != digest:
    raise RuntimeError("C04 byte-for-byte Drive derivative readback mismatch")
print("[A9-CWO C04 DERIVATIVE PILOT | 05/06] READBACK — derivative bytes confirmed, preparing immutable sidecar", flush=True)

sidecar = {
    "schema": "a9-cwo-provider-derivative-receipt-v1",
    "provenance_class": "PROVIDER_DERIVATIVE_NOT_ORIGINAL",
    "classification": "UNVERIFIED_ORIGINAL_MEDIA_FAILURE_OPEN",
    "source_run_id": handoff["run_id"], "guild_id": GUILD_ID,
    "channel_id": channel_id, "message_id": message_id,
    "attachment_id": attachment_id, "cdn_host": parts.hostname,
    "variant": "source_url", "declared_original_bytes": declared_bytes,
    "observed_derivative_bytes": len(raw), "declared_dimensions": declared_dims,
    "observed_dimensions": actual_dims, "sha256": digest,
    "derivative_file_id": image_file_id, "derivative_folder_id": folder_id,
    "does_not_certify_original": True, "does_not_advance_cursors": True,
    "does_not_update_handoff_or_frozen_receipt": True,
    "written_at_utc": datetime.now(timezone.utc).isoformat()
}
side_name = "C04_DERIVATIVE_RECEIPT__%s__%s.json" % (handoff["run_id"], attachment_id)
receipts_id = c03_find_child(run_root, "RUN_RECEIPTS")
prior = c04_match(receipts_id, side_name)
if prior:
    old_sidecar = json.loads(c04_read_bytes(prior["id"], max_bytes=256*1024).decode("utf-8"))
    if (old_sidecar.get("sha256") != digest or old_sidecar.get("derivative_file_id") != image_file_id):
        raise RuntimeError("C04 existing sidecar conflicts with current derivative; no overwrite")
    side_id = prior["id"]
else:
    side_obj = drive.files().create(
        body={"name": side_name, "parents": [receipts_id]},
        media_body=MediaIoBaseUpload(
            io.BytesIO((json.dumps(sidecar, sort_keys=True) + "\n").encode("utf-8")),
            mimetype="application/json", resumable=False),
        fields="id,name,size", supportsAllDrives=True).execute()
    side_id = side_obj["id"]
saved = json.loads(c04_read_bytes(side_id, max_bytes=256*1024).decode("utf-8"))
if saved.get("sha256") != digest or saved.get("provenance_class") != "PROVIDER_DERIVATIVE_NOT_ORIGINAL":
    raise RuntimeError("C04 readback provenance mismatch")
print("[A9-CWO C04 DERIVATIVE PILOT | RESULT] " + json.dumps({
    "status": "ONE_DERIVATIVE_SAVED_OR_READBACK_VERIFIED",
    "source_run_id": handoff["run_id"], "attachment_id": attachment_id,
    "declared_bytes": declared_bytes, "derivative_bytes": len(raw),
    "derivative_sha256": digest, "derivative_drive_id": image_file_id,
    "receipt_drive_id": side_id, "original_certified": False,
    "collector_cursors_advanced": False, "frozen_receipt_changed": False}, sort_keys=True), flush=True)
print("[A9-CWO C04 DERIVATIVE PILOT | 06/06] DONE — one provider derivative archived separately, original failure still OPEN", flush=True)


## C05 — DAILY MACHINE INDEX (private JSONL)

Run C01, then C02, then C05. C02 now rescans the previous seven days on every run; a stale verified cursor extends the range backward for missed-day catch-up. C05 requires a *new* receipt with the seven-day scan rule. It refuses an older two-day receipt.

C05 reads the latest frozen C02 run receipt and SHA-verifies both existing raw and candidate JSONL files. It creates one immutable JSONL pointer partition per **message creation date in Nepal time**, under the existing restricted Drive intake. Each compact row includes provider IDs, message version digest and line-number pointers to the original files. **No second copy of the message body is written.**

C05 then writes an immutable per-run JSON manifest with date partition IDs, counts and original receipt status. The index contains ONLY new or changed versions seen in the run; unchanged messages are not new facts. Deduplicate downstream by Discord channel + message ID + candidate SHA256 across runs.

C05 may index a PARTIAL receipt, but labels it PARTIAL, keeps all failed-media integrity debts, never edits C02 high-water state or earlier archive objects, and never claims project progress, engineering approval or an authorized daily construction schedule. Re-run C05 for the same run to verify and reuse exact archive bytes.

C03 and C04 remain optional diagnostic/pilot cells; they neither certify original media nor replace C05.

**r010 fail-closed rule:** C05 must run only after C02 prints FINAL RECEIPT following operator `COLLECT` in the SAME Colab runtime. C02 preview-only clears its confirmed-run marker, and C05 rejects that path and any stale Drive handoff even if an older seven-day receipt exists.


In [ ]:
# A9-CWO | C05 — DATE-PARTITIONED PRIVATE MACHINE INDEX (revision 20261009-010)
# Run C01 then C02. Read frozen C02 receipts, never modify source cursors.
import io, json, hashlib
from datetime import datetime, timezone
from collections import defaultdict

print("[A9-CWO C05 DAILY INDEX | 01/06] START", flush=True)
if not globals().get("C02_CONFIRMED_RUN_ID") or not globals().get("C02_CONFIRMED_RECEIPT_ID"):
    raise RuntimeError("C05 needs a confirmed C02 execution in this Colab runtime; preview-only or stale handoff cannot be indexed.")
needed = ("drive", "ROOT_DRIVE_ID", "RUN_FOLDER_NAME", "GUILD_ID", "TZ",
          "children", "directory", "byte_readback", "save_bytes", "jsonbytes")
missing = [name for name in needed if name not in globals()]
if missing:
    raise RuntimeError("C01/C02 setup missing: " + ", ".join(missing))
root = directory(ROOT_DRIVE_ID, RUN_FOLDER_NAME)
handoff_file = children(root, "handoff_current.json")
if not handoff_file:
    raise RuntimeError("C02 handoff not saved; no daily index may be claimed.")
handoff = json.loads(byte_readback(handoff_file["id"]).decode("utf-8"))
receipt_id = handoff.get("receipt_file_id")
if not receipt_id:
    raise RuntimeError("Handoff has no frozen receipt ID.")
receipt = json.loads(byte_readback(receipt_id).decode("utf-8"))
run_id = receipt.get("run_id")
if run_id != handoff.get("run_id") or receipt.get("guild_id") != GUILD_ID:
    raise RuntimeError("Handoff/receipt source identity mismatch.")
if run_id != C02_CONFIRMED_RUN_ID or receipt_id != C02_CONFIRMED_RECEIPT_ID:
    raise RuntimeError("C05 refuses a stale receipt: run C02 and type COLLECT in this same runtime.")
if receipt.get("scan_rule") != "ROLLING_SEVEN_DAYS_WITH_CATCHUP_EXTENSION" or receipt.get("replay_lookback_days") != 7:
    raise RuntimeError("Frozen C02 receipt uses an older window. Run the new C02 first.")
if receipt.get("status") not in ("PASS", "PARTIAL"):
    raise RuntimeError("C02 status cannot be indexed: " + str(receipt.get("status")))
if receipt.get("preflight_confirmed") is not True:
    raise RuntimeError("C02 receipt has no confirmed live preflight; cannot claim today's index.")
print("[A9-CWO C05 DAILY INDEX | 02/06] FROZEN_RECEIPT_OK run=%s status=%s" %
      (run_id, receipt["status"]), flush=True)

by_day = defaultdict(list)
seen = set()
for src in receipt.get("sources", []):
    raw = src.get("raw_jsonl")
    cand = src.get("candidate_jsonl")
    if not raw and not cand:
        continue
    if not isinstance(raw, dict) or not isinstance(cand, dict):
        raise RuntimeError("Incomplete raw/candidate source pair.")
    raw_bytes = byte_readback(raw["id"])
    cand_bytes = byte_readback(cand["id"])
    if (hashlib.sha256(raw_bytes).hexdigest() != raw.get("sha256") or
        hashlib.sha256(cand_bytes).hexdigest() != cand.get("sha256")):
        raise RuntimeError("Archived source JSONL differs from frozen receipt.")
    raw_lines, cand_lines = raw_bytes.splitlines(), cand_bytes.splitlines()
    if len(raw_lines) != len(cand_lines) or len(cand_lines) != int(src.get("new_or_changed_messages", -1)):
        raise RuntimeError("One immutable raw/candidate line count disagrees with receipt.")
    for line_no, (raw_line, cand_line) in enumerate(zip(raw_lines, cand_lines), 1):
        original, row = json.loads(raw_line), json.loads(cand_line)
        mid, cid = str(row["message_id"]), str(row["channel_id"])
        if (str(original["id"]) != mid or cid != str(src["channel_id"]) or
            str(row["guild_id"]) != GUILD_ID):
            raise RuntimeError("Source id mismatch on line %d in %s" % (line_no, cid))
        instant = datetime.fromisoformat(str(row["created_at"]).replace("Z", "+00:00"))
        if instant.tzinfo is None:
            raise RuntimeError("Discord timestamp missing timezone.")
        day = instant.astimezone(TZ).date().isoformat()
        digest = hashlib.sha256(jsonbytes(row)).hexdigest()
        key = (cid, mid, digest)
        if key in seen:
            raise RuntimeError("Duplicate source version within one run.")
        seen.add(key)
        by_day[day].append({
            "schema": "a9-cwo-discord-day-index-v1",
            "provider": "DISCORD", "run_id": run_id, "guild_id": GUILD_ID,
            "parent_channel_id": str(src["parent_channel_id"]),
            "channel_id": cid, "message_id": mid,
            "message_created_at_utc": instant.astimezone(timezone.utc).isoformat(),
            "date_npt": day, "event_version_sha256": digest,
            "source_message_url": "https://discord.com/channels/%s/%s/%s" % (GUILD_ID, cid, mid),
            "raw_drive_id": raw["id"], "raw_line_1_based": line_no,
            "candidate_drive_id": cand["id"], "candidate_line_1_based": line_no,
            "source_receipt_id": receipt_id,
            "attachment_count": len(original.get("attachments", [])),
            "evidence_class": "MESSAGE_METADATA_POINTER_ONLY",
            "admission_status": "NOT_RECONCILED",
            "attachments_original_certification": "SOURCE_PASS" if src.get("status") == "PASS" else "SOURCE_PARTIAL"
        })
print("[A9-CWO C05 DAILY INDEX | 03/06] VERIFIED_SOURCES rows=%d days=%d" %
      (len(seen), len(by_day)), flush=True)

index_root = directory(root, "DAILY_INDEX_JSONL")
partitions = []
for day in sorted(by_day):
    ordered = sorted(by_day[day], key=lambda x: (x["message_created_at_utc"], x["channel_id"], x["message_id"]))
    target = directory(index_root, day)
    name = "A9_CWO_DISCORD_DAY_INDEX__%s__%s.jsonl" % (day, run_id)
    saved = save_bytes(target, name, b"".join(jsonbytes(row) for row in ordered), "application/x-ndjson")
    partitions.append({"date_npt": day, "rows": len(ordered), "drive_id": saved["id"], "sha256": saved["sha256"]})
print("[A9-CWO C05 DAILY INDEX | 04/06] PRIVATE_DAILY_PARTITIONS_READBACK_PASS", flush=True)

start = datetime.fromisoformat(str(receipt["collection_started_at"]).replace("Z", "+00:00"))
manifest = {
    "schema": "a9-cwo-discord-daily-index-manifest-v1", "run_id": run_id,
    "run_npt_date": start.astimezone(TZ).date().isoformat(),
    "source_receipt_drive_id": receipt_id, "source_receipt_status": receipt["status"],
    "scan_rule": "ROLLING_SEVEN_DAYS_WITH_CATCHUP_EXTENSION",
    "source_scope": {
        "guild_id": GUILD_ID,
        "parent_channel_ids": [str(x) for x in receipt["target_parent_channels"]],
        "active_children_only": True, "archived_threads_covered": False
    },
    "rows_indexed": len(seen), "partitions": partitions,
    "media_completeness": "PASS" if receipt["status"] == "PASS" else "PARTIAL",
    "data_admission": "INDEX_ONLY_NOT_CONSTRUCTION_TRUTH"
}
receipts = directory(root, "RUN_RECEIPTS")
saved = save_bytes(receipts, "A9_CWO_DISCORD_DAILY_INDEX_MANIFEST__%s.json" % run_id,
                   jsonbytes(manifest), "application/json")
print("[A9-CWO C05 DAILY INDEX | 05/06] IMMUTABLE_MANIFEST_READBACK_PASS", flush=True)
print(json.dumps({"status": "INDEXED_WITH_SOURCE_" + receipt["status"],
                  "run_id": run_id, "rows_indexed": len(seen),
                  "partitions": partitions, "manifest_drive_id": saved["id"],
                  "frozen_receipt_drive_id": receipt_id,
                  "original_media_certified": receipt["status"] == "PASS",
                  "cursors_advanced_by_c05": False}, ensure_ascii=False, indent=2), flush=True)
print("[A9-CWO C05 DAILY INDEX | 06/06] PASS — private index; no fact admission.", flush=True)


## C06 — FULL HISTORY / RESUMABLE BACKFILL (separate from C02 seven-day maintenance)

Run **C01** first, then **C06** (C02 need not run). C06 prints every accessible guild message source and discoverable active/archived thread. Read its names, IDs and scope gaps; enter `FULL_HISTORY` to write a bounded, resumable batch or anything else for preview only.

C06 stores complete original Discord message objects as compressed lossless **gzip JSONL** by immutable page, records original-media attempts and unresolved/deferred cases separately, verifies each Drive object by SHA-256 readback, and resumes with an independent historical cursor. Repeat until ALL sources are exhausted, discovery gaps resolved and original-media recovery debt cleared. C06 does not publish to the website or send Slack messages.

**IMPORTANT:** A source that has all message pages but outstanding media is still NOT complete. Archived-thread permission failures and non-message channel types must appear as coverage gaps. Discord cannot recover messages already deleted or historical edit versions the bot never saw. Query-index generation and downstream publishing remain blocked until verified archive provenance and appropriate authorization.


In [ ]:
# A9-CWO | C06 — FULL GUILD HISTORY, RESUMABLE, ARCHIVE-FIRST (revision 20261009-013)
# C01 required; C02 and the daily seven-day cursor are NOT used or changed here.
# Discord API can only return messages still accessible to the authorized bot.
import io as h_io, os as h_os, re as h_re, json as h_json, gzip as h_gzip
import hashlib as h_hash, tempfile as h_temp, time as h_time
from datetime import datetime as h_dt, timezone as h_utc
from zoneinfo import ZoneInfo as h_ZoneInfo
import requests as h_requests
from googleapiclient.http import MediaIoBaseUpload as h_MemUpload, MediaFileUpload as h_FileUpload, MediaIoBaseDownload as h_Download

for h_required in ("drive", "BOT_TOKEN", "GUILD_ID", "ROOT_DRIVE_ID"):
    if h_required not in globals():
        raise RuntimeError("Run authorized C01 SETUP before C06: missing " + h_required)
H_TZ = h_ZoneInfo("Asia/Kathmandu")
H_API = "https://discord.com/api/v10"
H_RUN_AT = h_dt.now(h_utc.utc)
H_RUN_ID = H_RUN_AT.strftime("%Y%m%dT%H%M%SZ")
H_FULL_SCOPE = "ALL_ACCESSIBLE_GUILD_MESSAGE_CHANNELS_AND_THREADS"
H_MAX_PAGES_PER_INVOCATION = 8  # bounded, resumable; repeat C06 until all sources exhausted
H_MAX_MEDIA_ATTEMPTS_PER_INVOCATION = 20
H_MEDIA_MAX_BYTES = 100 * 1024 * 1024
H_DISCORD_EPOCH = 1420070400000
H_TOP = str((int(H_RUN_AT.timestamp() * 1000) - H_DISCORD_EPOCH) << 22)
H_HTTP = h_requests.Session()
H_HTTP.headers.update({"Authorization": "Bot " + BOT_TOKEN, "User-Agent": "A9-CWO-HistoricalCapture/1.0"})

def h_log(message):
    print("[A9-CWO C06 | %s NPT] %s" %
          (h_dt.now(H_TZ).strftime("%Y-%m-%d %H:%M:%S"), message), flush=True)

def h_payload(obj):
    return (h_json.dumps(obj, ensure_ascii=False, sort_keys=True, separators=(",", ":")) + "\n").encode("utf-8")

def h_req(path, params=None, allow_failure=False):
    for attempt in range(6):
        response = H_HTTP.get(H_API + path, params=params, timeout=90)
        if response.status_code == 429:
            delay = min(30.0, max(.2, float(response.json().get("retry_after", 1))))
            h_log("Discord rate limit: %s, %.1fs" % (path, delay))
            h_time.sleep(delay)
            continue
        if response.status_code >= 500 and attempt < 5:
            h_time.sleep(min(2 ** attempt, 16))
            continue
        if response.status_code >= 400:
            reason = "HTTP %d %s" % (response.status_code, path)
            if allow_failure:
                return None, reason
            raise RuntimeError(reason)
        return response.json(), None
    raise RuntimeError("Discord retries exhausted: " + path)

def h_child(parent, name):
    safe_name = name.replace("\\", "\\\\").replace("'", "\\'")
    files = []
    token = None
    while True:
        rows = drive.files().list(
            q="trashed=false and '%s' in parents and name='%s'" % (parent, safe_name),
            fields="nextPageToken,files(id,name,mimeType,size,md5Checksum)",
            pageToken=token, pageSize=100, supportsAllDrives=True,
            includeItemsFromAllDrives=True).execute()
        files.extend(rows.get("files", []))
        token = rows.get("nextPageToken")
        if not token: break
    if len(files) > 1: raise RuntimeError("Duplicate Drive artifact name: " + name)
    return files[0] if files else None

def h_folder(parent, name):
    match = h_child(parent, name)
    if match:
        if match["mimeType"] != "application/vnd.google-apps.folder":
            raise RuntimeError("Expected Drive folder: " + name)
        return match["id"]
    return drive.files().create(
        body={"name": name, "parents": [parent], "mimeType": "application/vnd.google-apps.folder"},
        fields="id", supportsAllDrives=True).execute()["id"]

def h_read(file_id):
    stream = h_io.BytesIO()
    loader = h_Download(stream, drive.files().get_media(
        fileId=file_id, supportsAllDrives=True))
    done = False
    while not done:
        _, done = loader.next_chunk()
    return stream.getvalue()

def h_immutable(parent, name, raw, mime="application/octet-stream"):
    digest = h_hash.sha256(raw).hexdigest()
    current = h_child(parent, name)
    if current:
        if h_hash.sha256(h_read(current["id"])).hexdigest() != digest:
            raise RuntimeError("Existing Drive immutable artifact has conflicting SHA-256: " + name)
        return {"id":current["id"], "sha256":digest, "bytes":len(raw), "reused":True}
    saved = drive.files().create(
        body={"name":name, "parents":[parent]},
        media_body=h_MemUpload(h_io.BytesIO(raw), mimetype=mime, resumable=True),
        fields="id,size", supportsAllDrives=True).execute()
    if h_hash.sha256(h_read(saved["id"])).hexdigest() != digest:
        raise RuntimeError("Drive SHA-256 readback failure: " + name)
    return {"id":saved["id"], "sha256":digest, "bytes":len(raw), "reused":False}

def h_mutable_json(folder, name, value):
    raw = h_payload(value)
    old = h_child(folder, name)
    media = h_MemUpload(h_io.BytesIO(raw), mimetype="application/json", resumable=True)
    if old:
        fid = drive.files().update(fileId=old["id"],media_body=media,
                                  fields="id",supportsAllDrives=True).execute()["id"]
    else:
        fid = drive.files().create(body={"name":name,"parents":[folder]},media_body=media,
                                  fields="id",supportsAllDrives=True).execute()["id"]
    if h_read(fid) != raw: raise RuntimeError("State readback failure: " + name)
    return fid

def h_archived(parent_id, subtype):
    if subtype == "joined":
        base = "/channels/%s/users/@me/threads/archived/private" % parent_id
    else:
        base = "/channels/%s/threads/archived/%s" % (parent_id,subtype)
    found, error, before, cursors = [], None, None, set()
    for _ in range(1000):
        params = {"limit":100}
        if before: params["before"] = before
        batch, error = h_req(base,params,allow_failure=True)
        if error: return found, error
        items = batch.get("threads", [])
        found.extend(items)
        if not batch.get("has_more"): return found, None
        if not items: return found, "Archived-thread provider reported has_more with empty page"
        # Discord archived public/private are paged by archive timestamp;
        # joined-private is paged by thread ID.
        if subtype == "joined":
            marker = str(items[-1]["id"])
        else:
            marker = str((items[-1].get("thread_metadata") or {}).get("archive_timestamp") or "")
        if not marker or marker in cursors:
            return found, "Archived-thread pagination did not advance"
        cursors.add(marker)
        before = marker
    return found, "Archived-thread discovery exceeded 1000 pages"

def h_discover():
    channels, error = h_req("/guilds/%s/channels" % GUILD_ID)
    if error: raise RuntimeError(error)
    sources = {}
    gaps = []
    # Parent channel messages plus ALL visible active/archived child threads.
    for channel in channels:
        cid, kind = str(channel["id"]), int(channel.get("type",-1))
        if kind in (0,5,15,16):
            sources[cid] = {
                "id":cid, "name":channel.get("name"), "parent_id":cid,
                "type":kind, "kind":"PARENT", "fetch_messages":kind in (0,5)}
        else:
            gaps.append({"channel_id":cid,"channel_name":channel.get("name"),
                         "type":kind,"reason":"UNSUPPORTED_NON_MESSAGE_CHANNEL_TYPE"})
    active, error = h_req("/guilds/%s/threads/active" % GUILD_ID, allow_failure=True)
    if error: gaps.append({"provider":"active_threads","reason":error})
    for thread in (active or {}).get("threads",[]):
        cid = str(thread["id"])
        parent = str(thread.get("parent_id"))
        if parent in sources:
            sources[cid] = {"id":cid,"name":thread.get("name"),"parent_id":parent,
                            "type":thread.get("type"),"kind":"ACTIVE_THREAD","fetch_messages":True}
    for pid, parent in list(sources.items()):
        if parent["kind"] != "PARENT": continue
        for subtype in ("public","private","joined"):
            threads, issue = h_archived(pid,subtype)
            if issue:
                gaps.append({"parent_id":pid,"parent_name":parent["name"],
                             "provider":"archived_"+subtype,"reason":issue})
            for thread in threads:
                tid = str(thread["id"])
                if tid not in sources:
                    sources[tid] = {"id":tid,"name":thread.get("name"),
                                    "parent_id":pid,"type":thread.get("type"),
                                    "kind":"ARCHIVED_THREAD","fetch_messages":True}
    return sorted(sources.values(),key=lambda x:(x["parent_id"],x["kind"],x["id"])),gaps

def h_media(folder, msg, attachment):
    aid, mid = str(attachment["id"]), str(msg["id"])
    declared = int(attachment.get("size") or 0)
    name = mid+"__"+aid+"__"+h_re.sub(r"[^-._a-zA-Z0-9]+","_",attachment.get("filename","file"))[:90]
    existing = h_child(folder,name)
    if existing and declared and int(existing.get("size") or -1) != declared:
        return {"attachment_id":aid,"status":"FAILED","reason":"DRIVE_ORIGINAL_SIZE_MISMATCH"}
    # An existing file's matching size alone is NOT original fidelity proof.
    # Re-fetch source CDN bytes and compare their SHA-256 to the Drive readback.
    url = str(attachment.get("url") or "")
    from urllib.parse import urlparse
    parsed = urlparse(url)
    if parsed.scheme != "https" or parsed.hostname not in ("cdn.discordapp.com","media.discordapp.net"):
        return {"attachment_id":aid,"status":"FAILED","reason":"INVALID_OR_NONCANONICAL_ATTACHMENT_URL"}
    if declared > H_MEDIA_MAX_BYTES:
        return {"attachment_id":aid,"status":"FAILED","reason":"DECLARED_MEDIA_OVER_SIZE_LIMIT"}
    try:
        # Do not forward the bot header to CDN; use a separate HTTP request.
        with h_requests.get(url,stream=True,timeout=120,
                            headers={"Accept-Encoding":"identity","User-Agent":"A9-CWO-HistoricalArchive"}) as response:
            if not response.ok: raise RuntimeError("CDN_HTTP_%d" % response.status_code)
            with h_temp.NamedTemporaryFile(suffix=".bin") as tmp:
                digest = h_hash.sha256()
                size = 0
                for block in response.iter_content(chunk_size=1024*1024):
                    if block:
                        size += len(block)
                        if size > H_MEDIA_MAX_BYTES: raise RuntimeError("MEDIA_BYTES_OVER_LIMIT")
                        digest.update(block)
                        tmp.write(block)
                tmp.flush()
                if declared and size != declared:
                    return {"attachment_id":aid,"status":"FAILED","reason":"ORIGINAL_SIZE_MISMATCH",
                            "discord_declared_bytes":declared,"cdn_observed_bytes":size}
                if existing:
                    stored=h_read(existing["id"])
                    if h_hash.sha256(stored).hexdigest() != digest.hexdigest():
                        return {"attachment_id":aid,"status":"FAILED",
                                "reason":"EXISTING_DRIVE_SHA_NOT_EQUAL_FRESH_PROVIDER_SOURCE"}
                    return {"attachment_id":aid,"status":"ORIGINAL_REUSED","drive_id":existing["id"],
                            "bytes":size,"sha256":digest.hexdigest(),"provider_refetched":True}
                uploaded = drive.files().create(
                    body={"name":name,"parents":[folder]},
                    media_body=h_FileUpload(tmp.name,mimetype="application/octet-stream",resumable=True),
                    fields="id,size,md5Checksum",supportsAllDrives=True).execute()
        if int(uploaded.get("size") or -1) != size:
            raise RuntimeError("DRIVE_FILE_SIZE_READBACK_MISMATCH")
        # Preserve SHA of downloaded bytes and Drive provider checksum evidence.
        if h_hash.sha256(h_read(uploaded["id"])).hexdigest() != digest.hexdigest():
            raise RuntimeError("DRIVE_MEDIA_SHA256_READBACK_MISMATCH")
        return {"attachment_id":aid,"status":"ORIGINAL_VERIFIED","drive_id":uploaded["id"],
                "sha256":digest.hexdigest(),"bytes":size}
    except Exception as exc:
        return {"attachment_id":aid,"status":"FAILED","reason":str(exc)[:180]}

def h_capture(sources, gaps, root_id):
    # The source state is independent of C02's rolling-window cursor.
    archive_root = h_folder(root_id,"FULL_HISTORY_ARCHIVE_V1")
    rawdir=h_folder(archive_root,"MESSAGE_PAGES_GZIP_JSONL")
    mediadir=h_folder(archive_root,"ATTACHMENT_BYTES")
    pagedir=h_folder(archive_root,"PAGE_RECEIPTS")
    rundir=h_folder(archive_root,"RUN_RECEIPTS")
    current=h_child(archive_root,"historical_state_current.json")
    state=(h_json.loads(h_read(current["id"]).decode("utf-8")) if current else
           {"schema":"a9-cwo-full-history-state-v1","guild_id":GUILD_ID,
            "initial_snapshot_top":H_TOP,"sources":{},"first_run":H_RUN_ID})
    if state.get("guild_id") != GUILD_ID or state.get("schema")!="a9-cwo-full-history-state-v1":
        raise RuntimeError("Historical state identity conflict")
    page_budget, media_budget = H_MAX_PAGES_PER_INVOCATION, H_MAX_MEDIA_ATTEMPTS_PER_INVOCATION
    processed = []
    for meta in sources:
        sid = meta["id"]
        status=state["sources"].setdefault(sid,{"before":state["initial_snapshot_top"],
                                               "complete":False,"pages":0,"media_debt":0})
        if not meta["fetch_messages"]:
            status["complete"]=True
            status["reason"]="PARENT_WITHOUT_DIRECT_MESSAGE_HISTORY; CHILD_THREADS_SCANNED"
            continue
        if status["complete"] or page_budget <= 0: continue
        before = str(status["before"])
        h_log("SCAN %s %s; parent=%s; before=%s; page allowance=%d" %
              (meta["kind"],meta["name"],meta["parent_id"],before,page_budget))
        messages,error=h_req("/channels/%s/messages"%sid,{"limit":100,"before":before},allow_failure=True)
        if error:
            status["last_error"]=error
            gaps.append({"source_id":sid,"reason":error})
            continue
        lines=b"".join(h_payload(m) for m in messages)
        zipped=h_gzip.compress(lines,compresslevel=6,mtime=0)
        sha=h_hash.sha256(zipped).hexdigest()
        stem=sid+"__BEFORE_"+before+"__SHA_"+sha[:24]
        # Persist lossless message metadata before attempting attachment downloads.
        raw=h_immutable(rawdir,stem+".jsonl.gz",zipped,"application/gzip")
        attachments=[]
        for msg in messages:
            for att in msg.get("attachments",[]):
                if media_budget<=0:
                    attachments.append({"attachment_id":str(att.get("id")),"message_id":str(msg["id"]),
                                        "status":"DEFERRED_MEDIA_BUDGET"})
                    continue
                media_budget-=1
                response=h_media(mediadir,msg,att)
                response["message_id"]=str(msg["id"])
                attachments.append(response)
        attachment_debt=sum(1 for row in attachments
                            if row["status"] not in ("ORIGINAL_VERIFIED","ORIGINAL_REUSED"))
        receipt={
            "schema":"a9-cwo-history-page-v1","run_id":H_RUN_ID,"source":meta,
            "before":before,"snapshot_top":state["initial_snapshot_top"],
            "captured_at_utc":h_dt.now(h_utc.utc).isoformat(),
            "first_message_id":str(messages[0]["id"]) if messages else None,
            "last_message_id":str(messages[-1]["id"]) if messages else None,
            "raw":raw,"raw_jsonl_sha256":h_hash.sha256(lines).hexdigest(),
            "uncompressed_bytes":len(lines),"message_count":len(messages),
            "attachments":attachments,"attachment_debt":attachment_debt,
            "end_of_history":len(messages)<100}
        page=h_immutable(pagedir,stem+".receipt.json",h_payload(receipt),"application/json")
        # Advance only after raw and page receipt have exact Drive byte readback.
        status["pages"]+=1
        status["media_debt"]+=attachment_debt
        status["before"]=str(min(int(x["id"]) for x in messages)) if messages else before
        status["complete"]=len(messages)<100
        status["last_page_receipt_id"]=page["id"]
        status["last_run_id"]=H_RUN_ID
        status.pop("last_error",None)
        h_mutable_json(archive_root,"historical_state_current.json",state)
        page_budget-=1
        processed.append({"source_id":sid,"source_name":meta["name"],"receipt_id":page["id"],
                          "messages":len(messages),"media_debt":attachment_debt,
                          "complete":status["complete"]})
        h_log("ARCHIVED page %s: messages=%d; media issues=%d; exhausted=%s; Drive receipt=%s"%
              (sid,len(messages),attachment_debt,status["complete"],page["id"]))
    all_complete=all(state["sources"].get(x["id"],{}).get("complete") for x in sources)
    media_debt=sum(int(x.get("media_debt",0)) for x in state["sources"].values())
    coverage="PASS" if all_complete and not gaps and not media_debt else "PARTIAL"
    summary={"schema":"a9-cwo-history-run-v1","run_id":H_RUN_ID,"guild_id":GUILD_ID,
             "scope":H_FULL_SCOPE,"started_at_utc":H_RUN_AT.isoformat(),
             "ended_at_utc":h_dt.now(h_utc.utc).isoformat(),"status":coverage,
             "source_count":len(sources),"sources_completed":sum(
                 bool(state["sources"].get(x["id"],{}).get("complete")) for x in sources),
             "media_debt_total":media_debt,"discovery_gaps":gaps,
             "processed_pages":processed,
             "next_action":"REPEAT_C06_UNTIL_ALL_SOURCES_AND_MEDIA_ARE_VERIFIED" if coverage!="PASS" else "BUILD_PRIVATE_SQLITE_AND_RECONCILE"}
    saved=h_immutable(rundir,"HISTORY_RUN__"+H_RUN_ID+".json",h_payload(summary),"application/json")
    h_log("FINAL: %s; completed=%d/%d; media debt=%d; discovery gaps=%d; pages=%d; receipt=%s" %
          (coverage,summary["sources_completed"],len(sources),media_debt,len(gaps),len(processed),saved["id"]))
    return summary

h_log("PREFLIGHT — discovering accessible guild channels, active and archived threads; no archive writes yet")
H_SOURCES,H_GAPS=h_discover()
h_log("DISCOVERED %d message sources; %d explicit visibility/discovery gaps" % (len(H_SOURCES),len(H_GAPS)))
for h_i,h_item in enumerate(H_SOURCES,1):
    h_log("%03d %s | %s | id=%s | parent=%s | %s"%
          (h_i,h_item["kind"],h_item["name"],h_item["id"],h_item["parent_id"],
           "https://discord.com/channels/%s/%s"%(GUILD_ID,h_item["id"])))
for h_gap in H_GAPS:
    h_log("SCOPE GAP: "+h_json.dumps(h_gap,ensure_ascii=False,sort_keys=True))
h_log("Historical scope is all bot-readable text/announcement/forum/media channels and their discoverable threads.")
h_log("This cannot recover already deleted messages, previous edit versions, restricted channels, DM history, or missing original CDN bytes.")
h_confirm=input("Type FULL_HISTORY to archive a bounded batch of the discovered sources (otherwise PREVIEW ONLY): ").strip()
if h_confirm=="FULL_HISTORY":
    h_root_meta=drive.files().get(fileId=ROOT_DRIVE_ID,fields="id,name,mimeType",supportsAllDrives=True).execute()
    if h_root_meta.get("mimeType")!="application/vnd.google-apps.folder":
        raise RuntimeError("A9 archive root identity verification failed")
    H_HISTORY_RESULT=h_capture(H_SOURCES,H_GAPS,ROOT_DRIVE_ID)
else:
    h_log("PREVIEW ONLY — no historical archive objects or cursors modified")


## C07 — Original-attachment recovery (repeat as needed)

Run C01 → C06 (to load its helpers) → C07. The historical JSONL pages and page receipts are immutable. C07 reads their failed/deferred attachment references, refreshes attachment URLs from original Discord message IDs, verifies exact downloaded bytes against provider-declared originals and Drive readback, saves immutable recovery receipts, and updates the separate recovery state without rewriting raw history. Enter `REPAIR_MEDIA` to execute up to 25 original-media attempts, or preview only. A CDN derivative is never counted as an original. If media is irretrievable, C09 publication remains blocked.


In [ ]:
# A9-CWO | C07 — REPAIR FAILED ORIGINAL MEDIA FROM IMMUTABLE C06 PAGES (r012)
# Requires C01 plus C06 definitions in this kernel. This NEVER edits frozen C06 page receipts.
for z in ("h_child","h_folder","h_read","h_req","h_media","h_payload","h_immutable","h_mutable_json"):
    if z not in globals(): raise RuntimeError("Run C01 and then C06 (preview or confirmed) to initialize C07: "+z)
import json as r_json
from datetime import datetime as r_dt, timezone as r_utc
R_BUDGET=25
def r_list(folder):
    pages, token=[],None
    while True:
        response=drive.files().list(q="trashed=false and '%s' in parents"%folder,
               pageSize=100, pageToken=token,fields="nextPageToken,files(id,name,size)",
               supportsAllDrives=True,includeItemsFromAllDrives=True).execute()
        pages+=response.get("files",[])
        token=response.get("nextPageToken")
        if not token: return sorted(pages,key=lambda x:x["name"])
R_BASE=h_child(ROOT_DRIVE_ID,"FULL_HISTORY_ARCHIVE_V1")
if not R_BASE: raise RuntimeError("No C06 historical Drive archive exists. Run C06 FULL_HISTORY first.")
R_ROOT=R_BASE["id"]
R_PAGES=h_child(R_ROOT,"PAGE_RECEIPTS")
R_MEDIA=h_child(R_ROOT,"ATTACHMENT_BYTES")
R_STATE=h_child(R_ROOT,"historical_state_current.json")
if not R_PAGES or not R_MEDIA or not R_STATE: raise RuntimeError("Historical archive missing required folders or state")
R_OLD=r_json.loads(h_read(R_STATE["id"]).decode("utf-8"))
if R_OLD.get("guild_id")!=GUILD_ID: raise RuntimeError("Historical archive guild mismatch")
R_ROWS=[]
for pfile in r_list(R_PAGES["id"]):
    page=r_json.loads(h_read(pfile["id"]).decode("utf-8"))
    src=page["source"]["id"]
    for att in page.get("attachments",[]):
        key=str(att["attachment_id"])
        if att["status"] not in ("ORIGINAL_VERIFIED","ORIGINAL_REUSED") and key not in R_OLD.get("recovered_media",{}):
            R_ROWS.append({"source_id":src,"message_id":str(att["message_id"]),
                           "attachment_id":key,"previous_status":att["status"],"page_receipt":pfile["id"]})
R_ROWS=list({x["attachment_id"]:x for x in R_ROWS}.values())
h_log("C07 PREFLIGHT: %d unrecovered attachment IDs; max %d attempts this run"%
      (len(R_ROWS),R_BUDGET))
for entry in R_ROWS[:R_BUDGET]:
    h_log("REPAIR CANDIDATE channel=%s message=%s attachment=%s (%s)"%
          (entry["source_id"],entry["message_id"],entry["attachment_id"],entry["previous_status"]))
r_answer=input("Type REPAIR_MEDIA to attempt the displayed items; otherwise preview only: ").strip()
if r_answer=="REPAIR_MEDIA":
    R_RECOVER_FOLDER=h_folder(R_ROOT,"RECOVERY_RECEIPTS")
    R_RECOVERED=R_OLD.setdefault("recovered_media",{})
    R_RESULTS=[]
    for entry in R_ROWS[:R_BUDGET]:
        aid=entry["attachment_id"]
        try:
            msg,error=h_req("/channels/%s/messages/%s"%(entry["source_id"],entry["message_id"]),allow_failure=True)
            if error: raise RuntimeError(error)
            attachment=next((a for a in msg.get("attachments",[]) if str(a["id"])==aid),None)
            if not attachment: raise RuntimeError("Original attachment no longer in provider message")
            attempt=h_media(R_MEDIA["id"],msg,attachment)
        except Exception as err:
            attempt={"attachment_id":aid,"status":"FAILED","reason":str(err)[:180]}
        item={"schema":"a9-cwo-history-recovery-v1","run_id":H_RUN_ID,"source_id":entry["source_id"],
              "message_id":entry["message_id"],"attachment_id":aid,
              "page_receipt_id":entry["page_receipt"],"attempt":attempt,
              "verified_at_utc":r_dt.now(r_utc.utc).isoformat()}
        raw=h_payload(item)
        key=aid+"__"+h_hash.sha256(raw).hexdigest()[:20]+".json"
        object=h_immutable(R_RECOVER_FOLDER,key,raw,"application/json")
        R_RESULTS.append({"attachment_id":aid,"status":attempt["status"],"receipt_id":object["id"]})
        if attempt["status"] in ("ORIGINAL_VERIFIED","ORIGINAL_REUSED"):
            R_RECOVERED[aid]={"source_id":entry["source_id"],"receipt_id":object["id"],
                               "original_drive_id":attempt["drive_id"]}
            src_state=R_OLD["sources"].get(entry["source_id"])
            if src_state: src_state["media_debt"]=max(0,int(src_state.get("media_debt",0))-1)
            h_mutable_json(R_ROOT,"historical_state_current.json",R_OLD)
        h_log("C07 MEDIA %s: %s"%(aid,attempt["status"]))
    R_REPORT={"schema":"a9-cwo-repair-run-v1","run_id":H_RUN_ID,"attempts":R_RESULTS,
              "unresolved_before":len(R_ROWS),"successes":sum(x["status"] in ("ORIGINAL_VERIFIED","ORIGINAL_REUSED") for x in R_RESULTS)}
    h_immutable(R_RECOVER_FOLDER,"REPAIR_RUN__"+H_RUN_ID+".json",h_payload(R_REPORT),"application/json")
    h_log("C07 FINAL: recovered %d originals from %d attempted; Drive-only receipts preserved"%
          (R_REPORT["successes"],len(R_RESULTS)))
else: h_log("C07 PREVIEW ONLY — no provider media fetched or Drive state changed")


## C08 — Private SQLite query snapshot (only after historical message coverage)

Run C01 → C06 (helpers loaded) → C08. C08 checks source coverage and archived-thread discovery, reads every immutable Drive page and SHA-256-verifies the compressed and decompressed JSONL, inserts original message versions plus normalized source/attachment/reaction keys into indexed SQLite, checks integrity and stores a SHA-256-verified private database snapshot plus manifest in Drive. This is a query cache; gzip JSONL and source media remain canonical. It does **not** publish content and cannot turn a failed original-media capture into success.


In [ ]:
# A9-CWO | C08 — BUILD PRIVATE NORMALIZED SQLITE FROM VERIFIED IMMUTABLE DRIVE JSONL (r012)
# Requires C01 + C06 definitions. Pure query derivative, not public-site or Slack publication.
for z in ("h_child","h_folder","h_read","h_payload","h_immutable","h_hash","h_gzip"):
    if z not in globals(): raise RuntimeError("Run C01 then C06 for C08 helper: "+z)
import json as q_json, sqlite3 as q_sqlite, tempfile as q_temp, os as q_os
from datetime import datetime as q_dt, timezone as q_utc
Q_ROOT_OBJ=h_child(ROOT_DRIVE_ID,"FULL_HISTORY_ARCHIVE_V1")
if not Q_ROOT_OBJ: raise RuntimeError("No historical raw Drive archive available")
Q_ROOT=Q_ROOT_OBJ["id"]
Q_STAT_FILE=h_child(Q_ROOT,"historical_state_current.json")
Q_PAGE_DIR=h_child(Q_ROOT,"PAGE_RECEIPTS")
Q_RAW_DIR=h_child(Q_ROOT,"MESSAGE_PAGES_GZIP_JSONL")
Q_RUN_DIR=h_child(Q_ROOT,"RUN_RECEIPTS")
if not all((Q_STAT_FILE,Q_PAGE_DIR,Q_RAW_DIR,Q_RUN_DIR)):
    raise RuntimeError("Historical Drive archive is incomplete")
Q_STATE=q_json.loads(h_read(Q_STAT_FILE["id"]).decode("utf-8"))
def q_children(folder):
    result,token=[],None
    while True:
        data=drive.files().list(q="trashed=false and '%s' in parents"%folder,
             fields="nextPageToken,files(id,name,size)",pageSize=100,pageToken=token,
             supportsAllDrives=True,includeItemsFromAllDrives=True).execute()
        result+=data.get("files",[])
        token=data.get("nextPageToken")
        if not token: return sorted(result,key=lambda item:item["name"])
Q_LATEST=sorted((x for x in q_children(Q_RUN_DIR["id"]) if x["name"].startswith("HISTORY_RUN__")),
                key=lambda x:x["name"])
if not Q_LATEST: raise RuntimeError("No verified history collection run receipt")
Q_RUN=q_json.loads(h_read(Q_LATEST[-1]["id"]).decode("utf-8"))
Q_COMPLETE=(Q_RUN.get("source_count",0)>0 and Q_RUN.get("sources_completed")==Q_RUN.get("source_count")
            and not Q_RUN.get("discovery_gaps") and
            all(x.get("complete") for x in Q_STATE.get("sources",{}).values()))
if not Q_COMPLETE:
    raise RuntimeError("Query build blocked: full retrievable message history not yet archived or discovery gaps remain")
Q_RECEIPT_FILES=q_children(Q_PAGE_DIR["id"])
h_log("C08 PREFLIGHT: %d immutable JSONL page receipts, %d completed sources, %d unresolved originals (SQL private only)"%
      (len(Q_RECEIPT_FILES),Q_RUN["sources_completed"],
       sum(int(x.get("media_debt",0)) for x in Q_STATE["sources"].values())))
q_answer=input("Type BUILD_SQLITE to verify every raw page and write private indexed SQLite (otherwise preview only): ").strip()
if q_answer=="BUILD_SQLITE":
    with q_temp.TemporaryDirectory() as working:
        filename=q_os.path.join(working,"discord_history.sqlite3")
        db=q_sqlite.connect(filename)
        db.executescript("""
        PRAGMA journal_mode=DELETE;
        PRAGMA synchronous=FULL;
        CREATE TABLE sources(channel_id TEXT PRIMARY KEY,parent_channel_id TEXT,kind TEXT,name TEXT,channel_type INTEGER);
        CREATE TABLE capture_pages(receipt_id TEXT PRIMARY KEY,channel_id TEXT,raw_drive_id TEXT,
                                   raw_gzip_sha256 TEXT,raw_jsonl_sha256 TEXT,message_count INTEGER);
        CREATE TABLE messages(channel_id TEXT NOT NULL,message_id TEXT NOT NULL,version_sha256 TEXT NOT NULL,
                              parent_channel_id TEXT,author_id TEXT,created_at_utc TEXT,edited_at_utc TEXT,
                              content TEXT,raw_json TEXT NOT NULL,PRIMARY KEY(channel_id,message_id,version_sha256));
        CREATE TABLE attachments(attachment_id TEXT NOT NULL,message_id TEXT NOT NULL,channel_id TEXT NOT NULL,
                                 filename TEXT,declared_bytes INTEGER,content_type TEXT,media_status TEXT,
                                 original_drive_id TEXT,PRIMARY KEY(attachment_id,channel_id,message_id));
        CREATE TABLE reactions(channel_id TEXT,message_id TEXT,version_sha256 TEXT,reaction_json TEXT);
        CREATE INDEX idx_messages_channel_date ON messages(channel_id,created_at_utc);
        CREATE INDEX idx_messages_author_date ON messages(author_id,created_at_utc);
        CREATE INDEX idx_messages_message_id ON messages(message_id);
        CREATE INDEX idx_attachments_message ON attachments(message_id);
        CREATE INDEX idx_sources_parent ON sources(parent_channel_id);
        """)
        q_pages,q_msgs=0,0
        for item in Q_RECEIPT_FILES:
            receipt=q_json.loads(h_read(item["id"]).decode("utf-8"))
            if receipt.get("schema")!="a9-cwo-history-page-v1":
                raise RuntimeError("Unknown history page receipt schema: "+item["id"])
            raw=receipt["raw"]
            zipped=h_read(raw["id"])
            if h_hash.sha256(zipped).hexdigest()!=raw["sha256"]:
                raise RuntimeError("Compressed Drive page SHA-256 mismatch: "+item["id"])
            original=h_gzip.decompress(zipped)
            if h_hash.sha256(original).hexdigest()!=receipt["raw_jsonl_sha256"]:
                raise RuntimeError("Decompressed raw JSONL SHA-256 mismatch: "+item["id"])
            lines=[ln for ln in original.splitlines() if ln]
            if len(lines)!=int(receipt["message_count"]):
                raise RuntimeError("Raw JSONL line count mismatch: "+item["id"])
            source=receipt["source"]
            db.execute("INSERT OR IGNORE INTO sources VALUES(?,?,?,?,?)",
                       (source["id"],source["parent_id"],source["kind"],source["name"],source["type"]))
            db.execute("INSERT OR IGNORE INTO capture_pages VALUES(?,?,?,?,?,?)",
                       (item["id"],source["id"],raw["id"],raw["sha256"],
                        receipt["raw_jsonl_sha256"],len(lines)))
            archived_att={str(a["attachment_id"]):a for a in receipt.get("attachments",[])}
            for ln in lines:
                msg=q_json.loads(ln)
                raw_str=ln.decode("utf-8")
                digest=h_hash.sha256(ln).hexdigest()
                chan=source["id"]; mid=str(msg["id"])
                db.execute("INSERT OR IGNORE INTO messages VALUES(?,?,?,?,?,?,?,?,?)",
                           (chan,mid,digest,source["parent_id"],str((msg.get("author") or {}).get("id") or ""),
                            msg.get("timestamp"),msg.get("edited_timestamp"),msg.get("content"),raw_str))
                for att in msg.get("attachments",[]):
                    aid=str(att["id"]); evidence=archived_att.get(aid,{})
                    recovered=Q_STATE.get("recovered_media",{}).get(aid)
                    media_status="RECOVERED_ORIGINAL" if recovered else evidence.get("status","NOT_ARCHIVED")
                    drive_id=(recovered or {}).get("original_drive_id") or evidence.get("drive_id")
                    db.execute("INSERT OR IGNORE INTO attachments VALUES(?,?,?,?,?,?,?,?)",
                               (aid,mid,chan,att.get("filename"),att.get("size"),
                                att.get("content_type"),media_status,drive_id))
                for reaction in msg.get("reactions",[]):
                    db.execute("INSERT INTO reactions VALUES(?,?,?,?)",
                               (chan,mid,digest,q_json.dumps(reaction,ensure_ascii=False,sort_keys=True)))
                q_msgs+=1
            q_pages+=1
        db.commit()
        db.execute("PRAGMA integrity_check")
        integrity=db.fetchone() if False else db.execute("PRAGMA integrity_check").fetchone()[0]
        if integrity!="ok": raise RuntimeError("SQLite integrity check failed: "+str(integrity))
        summary=db.execute("SELECT COUNT(*) FROM messages").fetchone()[0]
        db.close()
        with open(filename,"rb") as f: binary=f.read()
        # Query snapshot is always derived and replaceable; raw JSONL pages stay canonical.
        q_dir=h_folder(Q_ROOT,"QUERY_SNAPSHOTS")
        q_database=h_immutable(q_dir,"discord_history__"+H_RUN_ID+"__"+h_hash.sha256(binary).hexdigest()[:16]+".sqlite3",
                               binary,"application/vnd.sqlite3")
        q_report={"schema":"a9-cwo-sqlite-build-v1","run_id":H_RUN_ID,
                  "source_run_receipt_id":Q_LATEST[-1]["id"],"query_database":q_database,
                  "source_pages":q_pages,"raw_page_rows":q_msgs,"indexed_message_versions":summary,
                  "sqlite_integrity_check":"ok",
                  "unresolved_original_media":sum(int(x.get("media_debt",0)) for x in Q_STATE["sources"].values()),
                  "release_allowed":False,"built_at_utc":q_dt.now(q_utc.utc).isoformat()}
        q_manifest=h_immutable(q_dir,"QUERY_BUILD__"+H_RUN_ID+".json",h_payload(q_report),"application/json")
        h_log("C08 PASS PRIVATE SQLITE: %d pages, %d source rows, %d indexed versions; DB %s; manifest %s"%
              (q_pages,q_msgs,summary,q_database["id"],q_manifest["id"]))
else: h_log("C08 PREVIEW ONLY — no database built")


## C09 — Publication gate (read-only)

Run C01 → C06 (helpers loaded) → C09 after completing full historical capture, original-media recovery, and the private SQLite build. C09 checks immutable Drive history/query provenance, **requires PASS** (no unarchived sources, no missing originals, no discovery gaps), and prints either `PASS_DATA_VERIFIED_ONLY` or `BLOCKED` with exact reasons. Even if data passes, website and Slack publication still requires project identity reconciliation, public-safe approval and authorized send/readback. C09 itself never posts anything.


In [ ]:
# A9-CWO | C09 — READ-ONLY ARCHIVE FIRST PUBLICATION GATE (r012)
# No website, Slack, API send, credentials or public payloads are produced here.
for z in ("h_child","h_read","h_hash"):
    if z not in globals(): raise RuntimeError("Run C01 then C06 before C09 gate")
import json as g_json
G_ARCHIVE=h_child(ROOT_DRIVE_ID,"FULL_HISTORY_ARCHIVE_V1")
if not G_ARCHIVE: raise RuntimeError("BLOCKED — no full-history Drive archive")
G_ROOT=G_ARCHIVE["id"]
G_STATE=h_child(G_ROOT,"historical_state_current.json")
G_RUNS=h_child(G_ROOT,"RUN_RECEIPTS")
G_QUERY=h_child(G_ROOT,"QUERY_SNAPSHOTS")
if not G_STATE or not G_RUNS or not G_QUERY: raise RuntimeError("BLOCKED — required Drive source/query evidence not found")
g_state=g_json.loads(h_read(G_STATE["id"]).decode("utf-8"))
def g_files(fid):
    rows,token=[],None
    while True:
        out=drive.files().list(q="trashed=false and '%s' in parents"%fid,
           fields="nextPageToken,files(id,name)",pageSize=100,pageToken=token,
           supportsAllDrives=True,includeItemsFromAllDrives=True).execute()
        rows+=out.get("files",[])
        token=out.get("nextPageToken")
        if not token: return rows
g_runs=sorted((x for x in g_files(G_RUNS["id"]) if x["name"].startswith("HISTORY_RUN__")),key=lambda x:x["name"])
g_sql=sorted((x for x in g_files(G_QUERY["id"]) if x["name"].startswith("QUERY_BUILD__")),key=lambda x:x["name"])
if not g_runs or not g_sql: raise RuntimeError("BLOCKED — no source-history receipt or query build manifest")
g_last=g_json.loads(h_read(g_runs[-1]["id"]).decode("utf-8"))
g_build=g_json.loads(h_read(g_sql[-1]["id"]).decode("utf-8"))
g_fail=[]
if g_last.get("status")!="PASS":g_fail.append("LAST_HISTORY_RECEIPT_NOT_PASS")
if g_last.get("discovery_gaps"):g_fail.append("INCOMPLETE_SOURCE_OR_ARCHIVED_THREAD_DISCOVERY")
if g_last.get("source_count")!=g_last.get("sources_completed"):g_fail.append("HISTORY_PAGINATION_UNFINISHED")
if any(not x.get("complete") for x in g_state.get("sources",{}).values()):g_fail.append("SOURCE_CURSOR_UNFINISHED")
if sum(int(x.get("media_debt",0)) for x in g_state.get("sources",{}).values()):g_fail.append("ORIGINAL_MEDIA_RECOVERY_INCOMPLETE")
if g_build.get("source_run_receipt_id")!=g_runs[-1]["id"]:g_fail.append("QUERY_NOT_FROM_CURRENT_ARCHIVE_RECEIPT")
if g_build.get("sqlite_integrity_check")!="ok":g_fail.append("QUERY_DATABASE_INTEGRITY_UNVERIFIED")
g_db=g_build.get("query_database",{})
if not g_db.get("id") or h_hash.sha256(h_read(g_db["id"])).hexdigest()!=g_db.get("sha256"):
    g_fail.append("QUERY_DATABASE_DRIVE_READBACK_FAILED")
print("[A9-CWO C09] ARCHIVE_FIRST_GATE = %s"%("PASS_DATA_VERIFIED_ONLY" if not g_fail else "BLOCKED"),flush=True)
print("[A9-CWO C09] GUILD=%s | history receipt=%s | sqlite snapshot=%s"%
      (GUILD_ID,g_runs[-1]["id"],g_db.get("id")),flush=True)
print("[A9-CWO C09] BLOCKERS="+g_json.dumps(g_fail,ensure_ascii=False),flush=True)
if not g_fail:
    print("[A9-CWO C09] DATA READY. NEXT: independently reconcile identities and facts; validate source provenance; approve public-safe website changes and authorized Slack messages. THIS CELL DOES NOT PUBLISH.",flush=True)
else:
    print("[A9-CWO C09] WEBSITE/SLACK PUBLICATION IS BLOCKED. Resolve all listed archive/data issues first.",flush=True)
